# v2.2: the four next steps, implemented

v2.1's `full` run (5 seeds, 2.4M env steps) passed the pre-registered test: Progress+Tree(LOO) **0.51** and Progress+Tree **0.50** vs GraphGPO 0.39 at equal environment steps, also with a fixed number of updates per iteration. Its open problem was sparse graphs (merge noise 0.5 / 0.9: Progress+Tree(LOO) 0.13 / 0.07 vs GraphGPO 0.24 / 0.15). v2.2 works through the four next steps. The results quoted below come from runs made while building this notebook (1.2M env steps, 3 seeds, IQM final eval success); the notebook reproduces them in sections 5, 5b and 6b.

| Step | What v2.2 adds | Result |
|---|---|---|
| 1. Confirm | `MODE = 'replicate'`: the pre-registered comparison on **new task pools** (train/eval maps never used before) with **10 seeds**; nothing else runs. | Not run yet (~1.5–2 h on 2 CPUs). |
| 2. Merge rate in the target environment | ALFWorld measured with GraphGPO's own state strings; section **5b** places it on the toy's noise scale. | Shared steps: 0.44 → 0.91 (exact), 0.68 → 0.93 (similarity merging) from a random to a 95%-expert policy ≈ toy noise 0.46 → 0 and 0.21 → 0. |
| 3. Sparse graphs: remedies and branch budget | Section **6b**: branch points spread along trajectories, a learned critic for unmerged states, 8 instead of 32 branch points (K = 1 or 4), credit only (no tree). | **No remedy closes the gap**, and the crossover is early (table below). |
| 4. Port | `ProgressTree.zip`: a verl-agent recipe (GraphGPO, Progress+Tree, Tree-GRPO proxy, attribution arms) with state restore by action replay, equal-GPU-hours budget guard, analysis script; `ProgressTree_ALFWorld_Colab.ipynb` runs it on one GPU. | Verified on CPU against real ALFWorld (mock LLM): estimator identical to this notebook's, every restore exact, branch prompts identical to their parents'. GPU run not done. |

**Where v2 stops winning (toy, IQM success at 1.2M env steps, 3 seeds):**

| merge noise (shared steps) | 0 (≈0.85) | 0.15 (≈0.71) | 0.3 (≈0.58) | 0.5 (≈0.39) | 0.9 (≈0.06) |
|---|---|---|---|---|---|
| GraphGPO | 0.308 | 0.294 | 0.297 | 0.236 | 0.161 |
| Progress+Tree(LOO), 32 branch points/group | **0.351** | 0.234 | 0.201 | 0.128 | 0.067 |
| same, 8 branch points/group | 0.227 | 0.180 | 0.153 | 0.114 | 0.091 |
| same, spread along trajectories | 0.253 | — | — | 0.102 | 0.084 |
| Progress+Tree(critic) | 0.348 | — | — | 0.118 | 0.031 |
| Progress(LOO), no tree | 0.133 | 0.115 | 0.107 | 0.101 | 0.137 |

- **v2's gain needs nearly complete merging.** It leads at noise 0 and is behind already at 0.15, while GraphGPO barely changes up to 0.3. A likely reason: at an unmerged state the pooled value is just the trajectory's own outcome, so the step into it takes the whole advantage, whereas GraphGPO's credit falls back to its episode term there.
- **The tree is what pays at noise 0, and only with a large budget**: 8 branch points per group (33% of env steps) lose to GraphGPO even without noise; 32 (61%) win. Without the tree, progress credit alone is the weakest arm when states merge.
- **None of the remedies helps**: spreading branch points costs 0.10 at noise 0 and gains nothing elsewhere; the critic matches the default at noise 0 and is worst at 0.9.
- **What this means for ALFWorld**: with similarity merging it sits at noise ≈ 0.1–0.2 for weak-to-good policies, i.e. on the losing side of the crossover for most of training. The toy's noise is uniform random while ALFWorld's is structured, and an LLM policy is less random than the probe's, so the port starts with a cheap pilot that measures the merge rate live before any multi-seed comparison.

**Modes:** `smoke` (~5 min), `fast` (~1 h), `full` (~3.5 h; adds the crossover noise levels to section 5), `replicate` (step 1 only). Times are for a 2-CPU Colab runtime.

## The v2 concept (unchanged)

This notebook tests the revised concept that came out of the v1 results (Hybrid II stages 1–3 + H).

| Piece | What it does |
|---|---|
| **Tree sampling** | Each task group = G fresh rollouts **plus one branch from each selected visited state** (states are restored; text environments can replay the action prefix). Every branch is training data. |
| **Merged-state value pooling** | Identical states are merged across all rollouts and branches (GraphGPO's graph). V(s) = success rate of every observed continuation through s. |
| **Progress credit** | Step advantage = V(next state) − V(state), scaled per group like GRPO. Along each rollout the advantages add up exactly to the GRPO advantage, so the outcome credit is moved to the steps where the success probability changed. No separate episode term. |
| **Leave-one-out variant** | `Progress+Tree(LOO)`: each step is compared against its state's value estimated *without* its own trajectory; where no other trajectory visited the state, the last available baseline along the trajectory is used. It equals progress credit when states merge, RLOO when they don't, and outcome minus branch-point value inside a branch. |
| **Branch-point selection** | `smart`: branch from distinct visited states that are not absorbing (the agent has moved out of them at least once) and not cached dead ends (never succeeded in ≥ 8 pooled tries while the task's start state succeeds ≥ 25% of the time). `random`: the same number of branch points drawn uniformly from trajectory positions (Tree-GRPO-style random expansion). |

**What changed from v1**
- v1's frontier MC re-linked frontier states into GraphGPO's graph and discarded its continuations. v2 replaces re-linking with progress credit and **trains on the branches**.
- All arms now run to the **same environment-step budget**, so curves overlap and equal env steps is the main test.
- Every arm uses the **same minibatch size** and trains on all its samples, so at equal environment steps all arms get the same training compute and about the same number of optimizer steps. A separate check (3e) repeats the key comparison with a fixed number of updates per iteration, the setting of trainers that size minibatches per prompt.
- **Attribution arms** separate the effect of extra data from branching (`Tree+GRPO`, `Tree+GraphGPO`) from the effect of better credit (`Progress+Tree`), and the effect of choosing branch points (`Progress+Tree(random)`).
- The credit study adds **cosine alignment** between estimated and ground-truth advantages, a proxy for how well the policy-gradient direction is estimated.
- New **sparse-graph stress test**: states merge less often, as with LLM observation text.
- The cost model now charges a fixed **overhead per iteration** (weight sync, scheduling, waiting for the slowest rollout) on top of environment steps. Training compute per sample is the same for every arm, so it doesn't change the ranking at equal environment steps.

**Tests, in order:** kill test → main comparison at equal env steps → attribution → update-setting check → credit study → sparse-graph stress test → (optional) ablations, LLM track → auto-generated summary.

> **What to expect** (from a `full`-mode run: 5 seeds, 2.4M environment steps per run). Treat these as one run's results, not settled numbers.
> - **Main result (equal environment steps, same minibatch size):** IQM success Progress+Tree **0.50** and Progress+Tree(LOO) **0.51**, vs GraphGPO 0.39, GraphGPO(2×G) 0.37 and GRPO 0.37. Both progress variants pass the pre-registered test against GraphGPO and GraphGPO(2×G), with separated confidence intervals.
> - **Robust to minibatching at this budget:** with a fixed number of optimizer steps per iteration they still win at equal environment steps (+0.10 and +0.11). In `fast` mode (1.2M steps), that setting favored GraphGPO.
> - **Where the gain comes from:** tree sampling alone added nothing (−0.01). Progress credit added +0.12 over graph credit on the same tree, and smart branch points added +0.09 over random ones (branch success 47% vs 26%). Outcome-only credit on a tree was the worst arm (0.27).
> - **Credit study:** gradient alignment 0.47 (Progress+Tree) and 0.45 (LOO) vs 0.28 for GraphGPO; on frontier steps LOO is best (0.43 vs 0.27 and 0.11).
> - **Sparse graphs (1.2M steps):** at merge noise 0.5 / 0.9, GraphGPO 0.24 / 0.15, Progress+Tree(LOO) 0.13 / 0.07, Progress+Tree 0.05 / 0.01. **v2's gains depend on states merging.**

## 0. Setup
The toy track runs on CPU on purpose (small MLPs and vectorized NumPy environments are faster there). The optional LLM track (section 7) needs a GPU. Approximate run times on a 2-CPU Colab runtime: `smoke` ~4 min, `fast` ~35 min, `full` ~2–2.5 h (more CPUs → proportionally faster).

In [ ]:
MODE = 'fast'   # 'smoke' | 'fast' | 'full' | 'replicate' (main comparison only: new task pools, 10 seeds)
import os, sys, time, json, math, warnings, multiprocessing as mp
warnings.filterwarnings('ignore', message='An input array is constant')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
try:
    import google.colab  # noqa
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
N_WORKERS = max(1, (os.cpu_count() or 2))
print(f"Colab: {IN_COLAB} | CPU workers: {N_WORKERS} | MODE: {MODE}")
if MODE == 'replicate':
    print("replicate: 5 arms x 10 seeds at 2.4M env steps on new task pools (~1.5-2 h on 2 CPUs); other sections are skipped")

## 1. Core library
Environment, policy, rollouts, credit estimators, tree branching, PPO, training loop and credit study. It's written to a module so that parallel worker processes can import it.

In [ ]:
%%writefile h2lab_core.py
# ===== h2lab v2: progress-credited tree rollouts on a merged state graph =====
# Environment, policy, rollouts, credit estimators (GRPO, GraphGPO, progress),
# frontier MC probes (v1), tree branching (v2), PPO, training loop, credit study.
import math, time, random
from collections import defaultdict, deque
import numpy as np
import torch
import torch.nn as nn

SUCCESS = -1          # sentinel graph node for the verified goal state
SALT_BASE = 10 ** 6   # graph-identity salt for the sparse-graph stress test; env state = key % SALT_BASE

# ---------------------------------------------------------------------------
# 1. Environment: "KeyDoor-Traps", a multi-turn task with irreversible mistakes
# ---------------------------------------------------------------------------
# A wall column splits a left room from a right room; a locked door sits in that
# wall. The agent starts in the left room and must pick up the correct key K, walk
# through the door and reach the goal G. Irreversible mistakes create dead ends
# (like a destructive tool call): picking up the wrong key X first (inventory is
# taken forever) or stepping on a trap (agent frozen). Episodes run to the horizon.
MOVES = np.array([[-1, 0], [1, 0], [0, -1], [0, 1]])  # up, down, left, right
ACTION_NAMES = ["up", "down", "left", "right"]


class KeyDoorTasks:
    def __init__(self, n_tasks, size=9, n_traps=2, seed=0):
        rng = np.random.default_rng(seed)
        S = size
        self.S, self.N = S, n_tasks
        self.wall = np.zeros((n_tasks, S, S), bool)
        self.trap = np.zeros((n_tasks, S, S), bool)
        self.door = np.zeros((n_tasks, 2), int)
        self.key = np.zeros((n_tasks, 2), int)
        self.bad = np.zeros((n_tasks, 2), int)
        self.goal = np.zeros((n_tasks, 2), int)
        self.start = np.zeros((n_tasks, 2), int)
        wc = S - 3
        for i in range(n_tasks):
            w = self.wall[i]
            w[0, :] = w[-1, :] = w[:, 0] = w[:, -1] = True
            w[:, wc] = True
            dr = rng.integers(1, S - 1)
            w[dr, wc] = False
            self.door[i] = (dr, wc)
            left = [(r, c) for r in range(1, S - 1) for c in range(1, wc)]
            right = [(r, c) for r in range(1, S - 1) for c in range(wc + 1, S - 1)]
            left = [p for p in left if p != (dr, wc - 1)]  # keep the cell in front of the door free
            picks = rng.choice(len(left), size=3 + n_traps, replace=False)
            cells = [left[j] for j in picks]
            self.start[i], self.key[i], self.bad[i] = cells[0], cells[1], cells[2]
            for (r, c) in cells[3:]:
                self.trap[i, r, c] = True
            self.goal[i] = right[rng.integers(len(right))]
        st = np.zeros((n_tasks, 4, S, S), np.float32)
        st[:, 0] = self.wall
        st[:, 1] = self.trap
        ar = np.arange(n_tasks)
        st[ar, 2, self.goal[:, 0], self.goal[:, 1]] = 1
        st[ar, 3, self.door[:, 0], self.door[:, 1]] = 1
        self.static_obs = st
        self.obs_dim = 7 * S * S + 4


def encode_key(r, c, inv, door, trapped, S):
    """Exact state hash used for graph-node merging."""
    return ((((r * S) + c) * 3 + inv) * 2 + door) * 2 + trapped


def decode_key(k, S):
    k = np.asarray(k, np.int64) % SALT_BASE  # strip a graph-identity salt, if any
    trapped = k % 2; k = k // 2
    door = k % 2; k = k // 2
    inv = k % 3; k = k // 3
    c = k % S; r = k // S
    return r, c, inv, door, trapped


class VecEnv:
    """Vectorized, state-restorable environment."""

    def __init__(self, tasks: KeyDoorTasks, task_ids, horizon, init_keys=None, init_t=None):
        self.T = tasks
        self.S = tasks.S
        self.task = np.asarray(task_ids, int)
        B = len(self.task)
        self.H = horizon
        if init_keys is None:
            st = tasks.start[self.task]
            self.r, self.c = st[:, 0].copy(), st[:, 1].copy()
            self.inv = np.zeros(B, int); self.door = np.zeros(B, int); self.trapped = np.zeros(B, int)
        else:
            r, c, inv, door, tr = decode_key(init_keys, self.S)
            self.r, self.c, self.inv, self.door, self.trapped = (np.asarray(x, int).copy() for x in (r, c, inv, door, tr))
        self.t = np.zeros(B, int) if init_t is None else np.asarray(init_t, int).copy()
        self.done = self.t >= self.H
        self.success = np.zeros(B, bool)

    def keys(self):
        return encode_key(self.r, self.c, self.inv, self.door, self.trapped, self.S).astype(np.int64)

    def obs(self):
        T, S, B = self.T, self.S, len(self.task)
        o = np.zeros((B, 7, S, S), np.float32)
        o[:, :4] = T.static_obs[self.task]
        ar = np.arange(B)
        o[:, 3] *= (1 - self.door)[:, None, None]
        kp = self.inv != 1; bp = self.inv != 2
        o[ar[kp], 4, T.key[self.task[kp], 0], T.key[self.task[kp], 1]] = 1
        o[ar[bp], 5, T.bad[self.task[bp], 0], T.bad[self.task[bp], 1]] = 1
        o[ar, 6, self.r, self.c] = 1
        extra = np.zeros((B, 4), np.float32)
        extra[ar, self.inv] = 1
        extra[:, 3] = self.trapped
        return np.concatenate([o.reshape(B, -1), extra], 1)

    def step(self, a):
        T = self.T
        live = ~self.done
        mv = live & (self.trapped == 0)
        nr = self.r + MOVES[a, 0] * mv
        nc = self.c + MOVES[a, 1] * mv
        is_door = (nr == T.door[self.task, 0]) & (nc == T.door[self.task, 1])
        blocked = T.wall[self.task, nr, nc]
        need_key = is_door & (self.door == 0)
        can_open = need_key & (self.inv == 1)
        blocked |= need_key & ~can_open
        self.door = np.where(can_open & mv, 1, self.door)
        self.r = np.where(blocked, self.r, nr)
        self.c = np.where(blocked, self.c, nc)
        on_trap = T.trap[self.task, self.r, self.c] & mv
        self.trapped = np.where(on_trap, 1, self.trapped)
        on_k = (self.r == T.key[self.task, 0]) & (self.c == T.key[self.task, 1]) & (self.inv == 0) & mv
        on_x = (self.r == T.bad[self.task, 0]) & (self.c == T.bad[self.task, 1]) & (self.inv == 0) & mv
        self.inv = np.where(on_k, 1, np.where(on_x, 2, self.inv))
        at_goal = (self.r == T.goal[self.task, 0]) & (self.c == T.goal[self.task, 1]) & live
        rew = at_goal.astype(np.float32)
        self.success |= at_goal
        self.t += live
        self.done |= at_goal | (self.t >= self.H)
        return rew, live


# ---------------------------------------------------------------------------
# 2. Policy
# ---------------------------------------------------------------------------
class MLPPolicy(nn.Module):
    def __init__(self, obs_dim, n_actions=4, hidden=256):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(obs_dim, hidden), nn.ReLU(),
                                 nn.Linear(hidden, hidden), nn.ReLU(),
                                 nn.Linear(hidden, n_actions))

    def forward(self, x):
        return self.net(x)

    @torch.no_grad()
    def act(self, obs_np, device):
        logits = self.forward(torch.as_tensor(obs_np, device=device))
        dist = torch.distributions.Categorical(logits=logits)
        a = dist.sample()
        return a.cpu().numpy(), dist.log_prob(a).cpu().numpy()


# ---------------------------------------------------------------------------
# 3. Rollouts (main rollouts, MC probes and tree branches share this)
# ---------------------------------------------------------------------------
def _pad_time(x, L, is_keys):
    if x.shape[0] >= L:
        return x
    fill = np.repeat(x[-1:], L - x.shape[0], 0) if is_keys else np.zeros((L - x.shape[0],) + x.shape[1:], x.dtype)
    return np.concatenate([x, fill], 0)


def rollout(tasks, task_ids, policy, horizon, device, init_keys=None, first_action=None,
            keep_obs=True, chunk=8192, init_t=None, merge_noise=0.0):
    """Run envs to completion; returns [T, B] arrays (keys are [T+1, B]).
    init_keys: restore these states (keys[0] keeps the given, possibly salted, key).
    first_action: forced first action (ground-truth Q). merge_noise: probability that a
    visited state gets a unique graph identity (sparse-graph stress test)."""
    B = len(task_ids)
    if B > chunk:
        parts = [rollout(tasks, task_ids[i:i + chunk], policy, horizon, device,
                         None if init_keys is None else init_keys[i:i + chunk],
                         None if first_action is None else first_action[i:i + chunk],
                         keep_obs, chunk, None if init_t is None else init_t[i:i + chunk], merge_noise)
                 for i in range(0, B, chunk)]
        Tmax = max(p['actions'].shape[0] for p in parts)
        out = {}
        for k in parts[0]:
            if k == 'env_steps':
                out[k] = sum(p[k] for p in parts)
            elif k in ('success', 'task'):
                out[k] = np.concatenate([p[k] for p in parts])
            else:
                out[k] = np.concatenate([_pad_time(p[k], Tmax + 1 if k == 'keys' else Tmax, k == 'keys') for p in parts], 1)
        return out
    env = VecEnv(tasks, task_ids, horizon, init_keys, init_t)
    keys = [np.asarray(init_keys, np.int64).copy() if init_keys is not None else env.keys()]
    acts, logps, actives, rews, obs_l = [], [], [], [], []
    for t in range(horizon):
        o = env.obs()
        a, lp = policy.act(o, device)
        if t == 0 and first_action is not None:
            a = np.asarray(first_action, int)
            with torch.no_grad():
                lg = policy(torch.as_tensor(o, device=device))
                lp = torch.log_softmax(lg, -1).gather(1, torch.as_tensor(a, device=device)[:, None])[:, 0].cpu().numpy()
        r, live = env.step(a)
        k = env.keys()
        if merge_noise > 0:
            salt = (np.random.random(B) < merge_noise) * np.random.randint(1, SALT_BASE, B)
            k = k + SALT_BASE * salt.astype(np.int64)
        k = np.where(r > 0, SUCCESS, k)
        k = np.where(live, k, keys[-1])
        keys.append(k); acts.append(a); logps.append(lp); actives.append(live); rews.append(r)
        if keep_obs:
            obs_l.append(o)
        if env.done.all():
            break
    out = dict(keys=np.stack(keys), actions=np.stack(acts), logp=np.stack(logps),
               active=np.stack(actives), reward=np.stack(rews),
               success=env.success.copy(), task=np.asarray(task_ids), env_steps=int(np.stack(actives).sum()))
    if keep_obs:
        out['obs'] = np.stack(obs_l)
    return out


def obs_concat(main, extra):
    """Observations of main + extra columns, time-padded (needed by the critic credit)."""
    T = main['active'].shape[0]
    return np.concatenate([main['obs'], _pad_time(extra['obs'], T, False)[:T]], 1)


def light_concat(main, extra, g_main, g_extra):
    """Columns of main + extra (keys/active/success/task only) for credit computation."""
    T = main['active'].shape[0]
    comb = dict(keys=np.concatenate([main['keys'], _pad_time(extra['keys'], T + 1, True)[:T + 1]], 1),
                active=np.concatenate([main['active'], _pad_time(extra['active'], T, False)[:T]], 1),
                success=np.concatenate([main['success'], extra['success']]),
                task=np.concatenate([main['task'], extra['task']]))
    return comb, np.concatenate([g_main, g_extra])


# ---------------------------------------------------------------------------
# 4. Graph construction and distances (GraphGPO)
# ---------------------------------------------------------------------------
def build_reverse_graph(transitions):
    rev = defaultdict(set)
    for s, sn in transitions:
        rev[sn].add(s)
    return rev


def distances_to_success(rev, nodes):
    """Unit-cost reverse BFS from SUCCESS (= the official Dijkstra with weight 1)."""
    d = {n: math.inf for n in nodes}
    d[SUCCESS] = 0
    q = deque([SUCCESS])
    while q:
        u = q.popleft()
        for p in rev.get(u, ()):
            if d.get(p, math.inf) == math.inf:
                d[p] = d[u] + 1
                q.append(p)
    return d


def fill_unreachable(d):
    """Official GraphGPO behaviour: unreachable -> (max finite distance) + 1."""
    finite = [v for v in d.values() if v != math.inf]
    far = max(finite) if finite else 0
    return {k: (far + 1 if v == math.inf else v) for k, v in d.items()}, far


def traj_transitions(batch, idx):
    K, act = batch['keys'], batch['active']
    out = []
    for b in idx:
        for t in range(act.shape[0]):
            if act[t, b]:
                out.append((int(K[t, b]), int(K[t + 1, b])))
    return out


# ---------------------------------------------------------------------------
# 5. Credit estimators
# ---------------------------------------------------------------------------
def _norm_groups(values, groups, eps=1e-6):
    """Mean/std-normalize within groups (singleton groups -> 0), as verl-agent does."""
    out = np.zeros_like(values, dtype=np.float64)
    buckets = defaultdict(list)
    for i, g in enumerate(groups):
        buckets[g].append(i)
    for g, ix in buckets.items():
        v = values[ix]
        out[ix] = 0.0 if len(ix) == 1 else (v - v.mean()) / (v.std(ddof=1) + eps)
    return out


def grpo_advantages(batch, group):
    """Outcome advantage (R - mean_group) / std_group, broadcast to every step."""
    A_traj = _norm_groups(batch['success'].astype(np.float64), group)
    return np.broadcast_to(A_traj[None, :], batch['active'].shape) * batch['active']


def graph_step_returns(batch, group, extra_transitions=None, gamma=0.1):
    """GraphGPO step return 10*gamma^d(next state) per task group (+ optional
    extra transitions from MC probes = graph completion). Returns returns and the
    raw (pre-fill) distance dict per group."""
    T, B = batch['active'].shape
    ret = np.zeros((T, B))
    raw_d = {}
    for g in np.unique(group):
        idx = np.where(group == g)[0]
        trans = traj_transitions(batch, idx)
        if extra_transitions and g in extra_transitions:
            trans = trans + extra_transitions[g]
        nodes = {s for s, _ in trans} | {sn for _, sn in trans}
        d = distances_to_success(build_reverse_graph(trans), nodes)
        raw_d[g] = d
        dfill, _ = fill_unreachable(d)
        for b in idx:
            for t in range(T):
                if batch['active'][t, b]:
                    ret[t, b] = 10.0 * gamma ** dfill[int(batch['keys'][t + 1, b])]
    return ret, raw_d


def graphgpo_advantages(batch, group, step_returns, w_step=1.0, w_ep=1.0):
    """Step advantage normalized within (task, anchor-state) groups + episode
    advantage normalized across the steps of the task group (official recipe)."""
    T, B = batch['active'].shape
    tt, bb = np.nonzero(batch['active'])
    anchor = batch['keys'][tt, bb]
    grp = group[bb]
    a_step = _norm_groups(step_returns[tt, bb], [(int(g), int(s)) for g, s in zip(grp, anchor)])
    a_ep = _norm_groups(batch['success'][bb].astype(np.float64), [int(g) for g in grp])
    A = np.zeros((T, B))
    A[tt, bb] = w_step * a_step + w_ep * a_ep
    return A


# --- progress credit: on-policy value differences on the merged graph ---------
# V(s) = success rate of every observed continuation through s (first visit per
# trajectory, branch or probe), pooled over merged states. Step advantage =
# V(s_{t+1}) - V(s_t) with V(SUCCESS) = 1 and a timed-out final step valued 0.
# Along a trajectory the advantages telescope to R - V(start): for main rollouts,
# the GRPO advantage, redistributed to the steps where success probability changed.
def _traj_states(keys_col, active_col):
    out = [int(keys_col[0])]
    for t in range(active_col.shape[0]):
        if not active_col[t]:
            break
        out.append(int(keys_col[t + 1]))
    return out


def value_stats(batch, group, probes=()):
    stats = defaultdict(dict)
    def add(g, states, outcome):
        seen = set()
        for s in states:
            if s == SUCCESS or s in seen:
                continue
            seen.add(s)
            st = stats[g].setdefault(s, [0.0, 0])
            st[0] += outcome; st[1] += 1
    for b in range(batch['active'].shape[1]):
        add(int(group[b]), _traj_states(batch['keys'][:, b], batch['active'][:, b]), float(batch['success'][b]))
    for rb in probes:
        for c in range(rb['active'].shape[1]):
            add(int(rb['group'][c]), _traj_states(rb['keys'][:, c], rb['active'][:, c]), float(rb['success'][c]))
    return stats


def _V(stats, g, s):
    if s == SUCCESS:
        return 1.0
    st = stats[g].get(s)
    return st[0] / st[1] if st else 0.0


def group_scale(batch, group, sigma_min):
    return {int(g): max(batch['success'][group == g].astype(float).std(ddof=1) if (group == g).sum() > 1 else 0.0, sigma_min)
            for g in np.unique(group)}


def progress_advantages(batch, group, scale, probes=()):
    stats = value_stats(batch, group, probes)
    T, B = batch['active'].shape
    A = np.zeros((T, B))
    for b in range(B):
        g = int(group[b])
        act = np.nonzero(batch['active'][:, b])[0]
        if len(act) == 0:
            continue
        for t in act:
            s = int(batch['keys'][t, b]); sn = int(batch['keys'][t + 1, b])
            q = 1.0 if sn == SUCCESS else (0.0 if t == act[-1] else _V(stats, g, sn))
            A[t, b] = (q - _V(stats, g, s)) / scale[g]
    return A


# --- v1 probes: MC continuations used only for credit ---------------------------
def progress_loo_advantages(batch, group, scale, probes=()):
    """Progress credit with leave-one-out baselines, robust to sparse merging.
    A_t = Q(t) - b(t): Q(t) = pooled V(s_{t+1}) (1 at success, 0 at timeout);
    b(t) = V(s_t) estimated WITHOUT this trajectory's own sample; if no other
    trajectory visited s_t, b(t) falls back to the last available baseline along
    this trajectory (ultimately the leave-one-out group mean at the start state).
    Dense merging -> close to progress credit; no merging -> RLOO spread over the
    unshared segment; inside a branch -> outcome minus the branch point's value."""
    stats = value_stats(batch, group, probes)
    T, B = batch['active'].shape
    A = np.zeros((T, B))
    for b in range(B):
        g = int(group[b]); R = float(batch['success'][b])
        act = np.nonzero(batch['active'][:, b])[0]
        base = None
        for t in act:
            s = int(batch['keys'][t, b]); sn = int(batch['keys'][t + 1, b])
            sm, n = stats[g].get(s, [0.0, 0])
            if n > 1:
                base = (sm - R) / (n - 1)
            elif base is None:
                base = 0.0
            q = 1.0 if sn == SUCCESS else (0.0 if t == act[-1] else _V(stats, g, sn))
            A[t, b] = (q - base) / scale[g]
    return A


class ValueNet(nn.Module):
    """Small learned value baseline V(obs) ~ P(success), trained online on observed outcomes."""
    def __init__(self, obs_dim, hidden=256):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(obs_dim, hidden), nn.ReLU(), nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, 1))

    def forward(self, x):
        return torch.sigmoid(self.net(x)[:, 0])


def _first_visit_targets(keys, active, success, obs):
    """(obs, outcome) pairs for the first visit of each state in each column."""
    X, y = [], []
    for b in range(active.shape[1]):
        seen = set()
        for t in range(active.shape[0]):
            if not active[t, b]:
                break
            k = int(keys[t, b])
            if k in seen:
                continue
            seen.add(k); X.append(obs[t, b]); y.append(float(success[b]))
    return (np.asarray(X, np.float32), np.asarray(y, np.float32)) if X else (None, None)


def train_critic(memory, obs_dim, keys, active, success, obs, cfg, device='cpu'):
    if memory.get('critic') is None:
        net = ValueNet(obs_dim, cfg['critic_hidden']).to(device)
        memory['critic'] = (net, torch.optim.Adam(net.parameters(), lr=cfg['critic_lr']))
    net, opt = memory['critic']
    X, y = _first_visit_targets(keys, active, success, obs)
    if X is None:
        return net
    Xt, yt = torch.as_tensor(X, device=device), torch.as_tensor(y, device=device)
    for _ in range(cfg['critic_epochs']):
        perm = torch.randperm(len(Xt), device=device)
        for i in range(0, len(Xt), 1024):
            ix = perm[i:i + 1024]
            loss = nn.functional.binary_cross_entropy(net(Xt[ix]), yt[ix])
            opt.zero_grad(); loss.backward(); opt.step()
    return net


def progress_hybrid_advantages(batch, group, scale, obs, critic, n_min=2, device='cpu'):
    """Progress credit with hybrid values: V(s) = pooled Monte Carlo success rate where
    at least n_min observed continuations passed through s (merged states), otherwise
    the learned critic's estimate. A_t = V(s_{t+1}) - V(s_t), with V(SUCCESS) = 1 and a
    timed-out final step valued 0, so each trajectory's advantages still telescope to
    R - V(start). Where states never merge, credit follows the critic's value changes
    instead of collapsing onto the first unshared step."""
    stats = value_stats(batch, group)
    T, B = batch['active'].shape
    with torch.no_grad():
        flat = torch.as_tensor(obs.reshape(-1, obs.shape[-1]), device=device)
        vc = torch.cat([critic(flat[i:i + 8192]) for i in range(0, len(flat), 8192)]).cpu().numpy().reshape(T, B)
    A = np.zeros((T, B))
    for b in range(B):
        g = int(group[b])
        act = np.nonzero(batch['active'][:, b])[0]
        def V(t):
            s = int(batch['keys'][t, b])
            st = stats[g].get(s)
            return st[0] / st[1] if (st and st[1] >= n_min) else float(vc[t, b])
        for t in act:
            sn = int(batch['keys'][t + 1, b])
            q = 1.0 if sn == SUCCESS else (0.0 if t == act[-1] else V(t + 1))
            A[t, b] = (q - V(t)) / scale[g]
    return A


def select_mc_nodes(batch, group, raw_d, mode='frontier', max_nodes=8, priority='branching', trigger='always'):
    """v1 stage-3 selection. mode='frontier': anchor states with no path to success;
    'all': every visited state. priority='branching': decision points first,
    self-loop-only states skipped; 'visits': most visited first.
    trigger: 'always' | 'mixed' (groups with >=1 success) | 'no_success'."""
    out = []
    T = batch['active'].shape[0]
    for g in np.unique(group):
        idx = np.where(group == g)[0]
        n_succ = int(batch['success'][idx].sum())
        if (trigger == 'mixed' and n_succ == 0) or (trigger == 'no_success' and n_succ > 0):
            continue
        cnt = defaultdict(int); succ = defaultdict(set)
        for b in idx:
            for t in range(T):
                if batch['active'][t, b]:
                    s = int(batch['keys'][t, b]); sn = int(batch['keys'][t + 1, b])
                    if mode == 'all' or raw_d[g].get(s, math.inf) == math.inf:
                        cnt[s] += 1
                        if sn != s:
                            succ[s].add(sn)
        if priority == 'branching':
            ranked = sorted([s for s in cnt if succ[s]], key=lambda s: (-len(succ[s]), -cnt[s]))[:max_nodes]
        else:
            ranked = sorted(cnt, key=lambda s: -cnt[s])[:max_nodes]
        out += [(int(g), int(batch['task'][idx[0]]), s) for s in ranked]
    return out


def mc_continuations(tasks, nodes, policy, K, horizon, device, merge_noise=0.0):
    """K on-policy continuations from each node (restored state). Returns graph
    transitions per group, the rollout batch (with 'group'), and its env-step cost."""
    if not nodes:
        return {}, None, 0
    rb = rollout(tasks, np.repeat([n[1] for n in nodes], K), policy, horizon, device,
                 init_keys=np.repeat(np.asarray([n[2] for n in nodes], np.int64), K), merge_noise=merge_noise)
    rb['group'] = np.repeat([n[0] for n in nodes], K)
    extra = defaultdict(list)
    for c in range(rb['active'].shape[1]):
        g = int(rb['group'][c])
        for t in range(rb['active'].shape[0]):
            if rb['active'][t, c]:
                extra[g].append((int(rb['keys'][t, c]), int(rb['keys'][t + 1, c])))
    return dict(extra), rb, rb['env_steps']


# --- v2 tree branching: branches are training data -----------------------------
def _dead_set(memory, task, cfg):
    """States that never succeeded in >= dead_min_n pooled tries (decayed), while the
    task's start state succeeds >= dead_task_rate of the time."""
    if memory is None:
        return set()
    cache = memory['cache'].get(task); s0 = memory['start'].get(task)
    if not cache or s0 not in cache:
        return set()
    sm0, n0 = cache[s0]
    if n0 <= 0 or sm0 / n0 < cfg['dead_task_rate']:
        return set()
    return {s for s, (sm, n) in cache.items() if n >= cfg['dead_min_n'] and sm <= 1e-9}


def update_memory(memory, comb, gcomb, cfg):
    stats = value_stats(comb, gcomb)
    for g, d in stats.items():
        idx = np.where(gcomb == g)[0]
        task = int(comb['task'][idx[0]])
        cache = memory['cache'].setdefault(task, {})
        for st in cache.values():
            st[0] *= cfg['dead_decay']; st[1] *= cfg['dead_decay']
        for s, (sm, n) in d.items():
            c = cache.setdefault(s, [0.0, 0.0]); c[0] += sm; c[1] += n
        memory['start'][task] = int(comb['keys'][0, idx[0]])


def shared_step_frac(batch, group):
    """Share of steps t >= 1 whose state is visited by >= 2 trajectories of its group
    (the same statistic as port/alfworld_merge_probe.py)."""
    shared = tot = 0
    for g in np.unique(group):
        idx = np.where(group == g)[0]
        vis = defaultdict(set); steps = []
        for b in idx:
            for t in range(batch['active'].shape[0]):
                if batch['active'][t, b]:
                    s = int(batch['keys'][t, b]); vis[s].add(b)
                    if t >= 1:
                        steps.append(s)
        tot += len(steps); shared += sum(1 for s in steps if len(vis[s]) >= 2)
    return shared / max(tot, 1)


def select_branch_nodes(batch, group, cfg, mode='smart', memory=None, rng=None):
    """Where to branch the group's tree.
    smart:  distinct visited states that are not absorbing (>=1 observed move out),
            not cached dead ends, most visited first, capped at branch_max_nodes.
    random: the same number of branch points drawn uniformly from the group's
            trajectory positions (Tree-GRPO-style random expansion)."""
    out, n_frozen, n_dead = [], 0, 0
    T = batch['active'].shape[0]
    rng = rng if rng is not None else np.random.default_rng(0)
    for g in np.unique(group):
        idx = np.where(group == g)[0]
        task = int(batch['task'][idx[0]])
        visits = defaultdict(int); moves = defaultdict(int); positions = []
        for b in idx:
            for t in range(T):
                if batch['active'][t, b]:
                    s = int(batch['keys'][t, b]); sn = int(batch['keys'][t + 1, b])
                    visits[s] += 1; positions.append(s)
                    if sn != s:
                        moves[s] += 1
        cand = [s for s in visits if s != SUCCESS]
        n_frozen += sum(1 for s in cand if moves[s] == 0)
        cand = [s for s in cand if moves[s] > 0]
        dead = _dead_set(memory, task, cfg)
        n_dead += sum(1 for s in cand if s in dead)
        cand = sorted([s for s in cand if s not in dead], key=lambda s: -visits[s])[:cfg['branch_max_nodes']]
        if mode == 'spread':
            # Evenly spaced points along each trajectory's distinct-state sequence (round-robin
            # over trajectories), skipping absorbing states and cached dead ends: covers the whole
            # horizon even when no state repeats.
            seqs = []
            for b in idx:
                seq = [s for s in _traj_states(batch['keys'][:, b], batch['active'][:, b])
                       if s != SUCCESS and moves[s] > 0 and s not in dead]
                seqs.append(list(dict.fromkeys(seq)))
            budget = len(cand); per = max(1, int(math.ceil(budget / max(len(seqs), 1)))); picked = []
            for seq in seqs:
                if seq:
                    pos = np.unique(np.linspace(0, len(seq) - 1, min(per, len(seq))).round().astype(int))
                    picked += [seq[p] for p in pos]
            cand = list(dict.fromkeys(picked))[:budget]
        if mode == 'random':
            cand = [positions[i] for i in rng.integers(len(positions), size=len(cand))] if positions else []
        out += [(int(g), task, s) for s in cand]
    return out, dict(branch_frozen_skipped=n_frozen, branch_dead_skipped=n_dead)


# ---------------------------------------------------------------------------
# 6. Arms
# ---------------------------------------------------------------------------
ARMS = {
    'GRPO':                  dict(credit='grpo'),
    'GRPO(2xG)':             dict(credit='grpo', gmult=2),
    'GraphGPO':              dict(credit='graph'),
    'GraphGPO(2xG)':         dict(credit='graph', gmult=2),
    'GraphGPO+FrontierMC':   dict(credit='graph', probes='frontier'),   # v1 concept (probes for credit only)
    'GraphGPO+VineAll':      dict(credit='graph', probes='all'),        # credit study: graph-channel ceiling
    'Progress':              dict(credit='progress'),                   # progress credit, no extra samples
    'Progress+VineAll':      dict(credit='progress', probes='all'),     # credit study: progress-channel ceiling
    'Progress+Tree':         dict(credit='progress', tree='smart'),     # v2 concept
    'Progress+Tree(random)': dict(credit='progress', tree='random'),    # branch-point ablation
    'Progress+Tree(LOO)':    dict(credit='progress_loo', tree='smart'), # v2.1: leave-one-out baselines (sparse-graph robust)
    'Progress+Tree(critic)': dict(credit='progress_hybrid', tree='smart'),   # v2.2: MC values where merged, learned critic elsewhere
    'Progress+Tree(LOO,spread)': dict(credit='progress_loo', tree='spread'),# v2.2: branch points spread along trajectories
    'Progress+Tree(critic,spread)': dict(credit='progress_hybrid', tree='spread'),
    'Progress(LOO)':         dict(credit='progress_loo'),               # no tree: drop-in credit change (first port step)
    'Progress(critic)':      dict(credit='progress_hybrid'),
    'Tree+GraphGPO':         dict(credit='graph', tree='smart'),        # attribution: same tree, graph credit
    'Tree+GRPO':             dict(credit='grpo', tree='smart'),         # attribution: same tree, outcome credit
}


def arm_spec(arm):
    return ARMS[arm]


def _credit(credit, batch, group, cfg, scale=None, probes=(), extra_transitions=None, obs=None, critic=None):
    if credit == 'progress_hybrid':
        return progress_hybrid_advantages(batch, group, scale, obs, critic, cfg['critic_n_min'])
    if credit == 'grpo':
        return grpo_advantages(batch, group)
    if credit == 'graph':
        ret, _ = graph_step_returns(batch, group, extra_transitions=extra_transitions, gamma=cfg['gamma'])
        return graphgpo_advantages(batch, group, ret, cfg['w_step'], cfg['w_ep'])
    if credit == 'progress':
        return progress_advantages(batch, group, scale, probes)
    if credit == 'progress_loo':
        return progress_loo_advantages(batch, group, scale, probes)
    raise ValueError(credit)


def compute_advantages(arm, batch, group, tasks, policy, cfg, device, memory=None, rng=None):
    """Returns (A for the main batch [T,B], extra training batch or None,
    extra env steps, diagnostics)."""
    spec = arm_spec(arm)
    credit, nm = spec['credit'], cfg.get('merge_noise', 0.0)
    diag = {}
    if credit == 'progress_hybrid' and memory is None:
        memory = dict(cache={}, start={})   # credit study: a critic fit on this batch only
    ret0, raw_d = graph_step_returns(batch, group, gamma=cfg['gamma'])
    tt, bb = np.nonzero(batch['active'])
    unl = np.array([raw_d[group[b]].get(int(batch['keys'][t, b]), math.inf) == math.inf for t, b in zip(tt, bb)])
    moving = batch['keys'][tt, bb] != batch['keys'][tt + 1, bb]
    diag['frontier_step_frac'] = float(unl.mean()) if len(unl) else 0.0
    diag['frontier_moving_step_frac'] = float(unl[moving].mean()) if moving.any() else 0.0
    diag['shared_step_frac'] = shared_step_frac(batch, group)
    scale = group_scale(batch, group, cfg['progress_sigma_min'])

    if spec.get('tree'):
        nodes, sk = select_branch_nodes(batch, group, cfg, spec['tree'], memory, rng)
        diag.update(sk)
        _, br, cost = mc_continuations(tasks, nodes, policy, cfg['branch_K'], cfg['mc_horizon'], device, nm)
        if br is None:
            critic = train_critic(memory, batch['obs'].shape[-1], batch['keys'], batch['active'], batch['success'],
                                  batch['obs'], cfg, device) if credit == 'progress_hybrid' else None
            return _credit(credit, batch, group, cfg, scale, obs=batch['obs'], critic=critic), None, 0, diag
        comb, gcomb = light_concat(batch, br, group, br['group'])
        obs_c = critic = None
        if credit == 'progress_hybrid':
            obs_c = obs_concat(batch, br)
            critic = train_critic(memory, obs_c.shape[-1], comb['keys'], comb['active'], comb['success'], obs_c, cfg, device)
        A_comb = _credit(credit, comb, gcomb, cfg, scale, obs=obs_c, critic=critic)
        B = batch['active'].shape[1]; Tb = br['active'].shape[0]
        extra = dict(obs=br['obs'], actions=br['actions'], logp=br['logp'], active=br['active'],
                     adv=A_comb[:Tb, B:])
        if memory is not None:
            update_memory(memory, comb, gcomb, cfg)
        diag.update(branch_nodes=len(nodes), branch_steps=int(cost), branch_success=float(br['success'].mean()))
        return A_comb[:, :B], extra, cost, diag

    if spec.get('probes'):
        mode = spec['probes']
        nodes = select_mc_nodes(batch, group, raw_d, mode=mode,
                                max_nodes=cfg['mc_max_nodes'] if mode == 'frontier' else cfg['vine_max_nodes'],
                                priority=cfg['mc_priority'] if mode == 'frontier' else 'visits',
                                trigger=cfg['mc_trigger'] if mode == 'frontier' else 'always')
        extra_t, rb, cost = mc_continuations(tasks, nodes, policy, cfg['mc_K'], cfg['mc_horizon'], device, nm)
        if credit == 'graph':
            ret2, raw2 = graph_step_returns(batch, group, extra_transitions=extra_t, gamma=cfg['gamma'])
            A = graphgpo_advantages(batch, group, ret2, cfg['w_step'], cfg['w_ep'])
            if nodes:
                diag['relink_rate'] = float(np.mean([raw2[g].get(s, math.inf) != math.inf for g, _, s in nodes]))
        else:
            A = progress_advantages(batch, group, scale, [rb] if rb is not None else [])
        diag['mc_nodes'] = len(nodes)
        return A, None, cost, diag

    if credit == 'graph':
        return graphgpo_advantages(batch, group, ret0, cfg['w_step'], cfg['w_ep']), None, 0, diag
    critic = train_critic(memory, batch['obs'].shape[-1], batch['keys'], batch['active'], batch['success'],
                          batch['obs'], cfg, device) if credit == 'progress_hybrid' else None
    return _credit(credit, batch, group, cfg, scale, obs=batch['obs'], critic=critic), None, 0, diag


# ---------------------------------------------------------------------------
# 7. PPO-clip update on per-step advantages
# ---------------------------------------------------------------------------
def flatten_for_update(batch, A):
    tt, bb = np.nonzero(batch['active'])
    return (batch['obs'][tt, bb], batch['actions'][tt, bb], batch['logp'][tt, bb], A[tt, bb])


def ppo_update(policy, opt, data, cfg, device):
    """update_mode='fixed_minibatch' (default): the same minibatch size for every arm.
    Every arm trains on all its samples for ppo_epochs, so at equal environment steps
    all arms get the same training compute AND about the same number of optimizer
    steps: the fair equal-compute comparison.
    'fixed_steps': n_minibatches per epoch whatever the data size, so every arm gets
    the same number of optimizer steps per iteration (arms with more samples per
    iteration get fewer, larger steps at equal env steps). This is what trainers that
    size minibatches per prompt do (e.g. verl scales the PPO minibatch by the number
    of responses per prompt), and it isolates per-update credit quality."""
    obs, act, logp_old, adv = [torch.as_tensor(x, device=device) for x in data]
    adv = adv.float(); logp_old = logp_old.float()
    n = len(obs)
    if n == 0:
        return
    mb = int(math.ceil(n / cfg['n_minibatches'])) if cfg.get('update_mode', 'fixed_steps') == 'fixed_steps' else cfg['minibatch']
    for _ in range(cfg['ppo_epochs']):
        perm = torch.randperm(n, device=device)
        for i in range(0, n, mb):
            ix = perm[i:i + mb]
            logits = policy(obs[ix])
            lp = torch.log_softmax(logits, -1).gather(1, act[ix].long()[:, None])[:, 0]
            ratio = torch.exp(lp - logp_old[ix])
            a = adv[ix]
            pg = -torch.min(ratio * a, torch.clamp(ratio, 1 - cfg['clip'], 1 + cfg['clip']) * a).mean()
            ent = torch.distributions.Categorical(logits=logits).entropy().mean()
            opt.zero_grad(); (pg - cfg['ent_coef'] * ent).backward()
            nn.utils.clip_grad_norm_(policy.parameters(), 1.0)
            opt.step()


@torch.no_grad()
def evaluate(tasks, task_ids, policy, horizon, device, reps=4):
    rb = rollout(tasks, np.repeat(task_ids, reps), policy, horizon, device, keep_obs=False)
    return float(rb['success'].mean())


# ---------------------------------------------------------------------------
# 8. Training loop for one (arm, seed)
# ---------------------------------------------------------------------------
DEFAULT_CFG = dict(
    size=9, n_traps=2, horizon=40, n_train_tasks=256, n_eval_tasks=128,
    tasks_per_iter=16, group_size=8,
    env_budget=None, max_iters=3000, eval_points=20,   # env-step budget mode (v2 default in the notebook)
    iters=200, eval_every=20,                          # iteration mode (used when env_budget is None)
    gamma=0.1, w_step=1.0, w_ep=1.0,
    mc_K=4, mc_max_nodes=8, mc_horizon=20, mc_priority='branching', mc_trigger='mixed', vine_max_nodes=64,
    branch_K=1, branch_max_nodes=32, dead_min_n=8, dead_task_rate=0.25, dead_decay=0.8,
    progress_sigma_min=0.2, merge_noise=0.0,
    critic_hidden=256, critic_lr=1e-3, critic_epochs=2, critic_n_min=2,
    task_seed_train=1000, task_seed_eval=2000,
    lr=3e-4, ppo_epochs=2, update_mode='fixed_minibatch', n_minibatches=5, minibatch=1024,
    clip=0.2, ent_coef=0.01, hidden=256,
)


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)


def train_arm(arm, seed, cfg, device='cpu', train_tasks=None, eval_tasks=None, verbose=False,
              policy_factory=None):
    set_seed(seed)
    train_tasks = train_tasks or KeyDoorTasks(cfg['n_train_tasks'], cfg['size'], cfg['n_traps'], seed=cfg.get('task_seed_train', 1000))
    eval_tasks = eval_tasks or KeyDoorTasks(cfg['n_eval_tasks'], cfg['size'], cfg['n_traps'], seed=cfg.get('task_seed_eval', 2000))
    policy = policy_factory() if policy_factory else MLPPolicy(train_tasks.obs_dim, hidden=cfg['hidden']).to(device)
    opt = torch.optim.Adam([p for p in policy.parameters() if p.requires_grad], lr=cfg['lr'])
    rng = np.random.default_rng(seed)
    memory = dict(cache={}, start={})
    budget = cfg.get('env_budget')
    marks = list(np.linspace(0, budget, cfg['eval_points'] + 1)[1:]) if budget else None
    max_iters = cfg['max_iters'] if budget else cfg['iters']
    G = cfg['group_size'] * arm_spec(arm).get('gmult', 1)
    env_steps, it, log, t0 = 0, 0, [], time.time()

    def do_eval():
        ev = evaluate(eval_tasks, np.arange(eval_tasks.N), policy, cfg['horizon'], device)
        log.append(dict(arm=arm, seed=seed, iter=it, env_steps=env_steps, eval_success=ev, wall=time.time() - t0))
        if verbose:
            print(f"[{arm}|s{seed}] it {it:4d} steps {env_steps:9d} eval {ev:.3f}")

    do_eval()
    while it < max_iters and not (budget and env_steps >= budget):
        tids = rng.choice(train_tasks.N, cfg['tasks_per_iter'], replace=False)
        ids = np.repeat(tids, G)
        group = np.repeat(np.arange(cfg['tasks_per_iter']), G)
        batch = rollout(train_tasks, ids, policy, cfg['horizon'], device, merge_noise=cfg.get('merge_noise', 0.0))
        A, extra, cost, diag = compute_advantages(arm, batch, group, train_tasks, policy, cfg, device, memory, rng)
        env_steps += batch['env_steps'] + cost
        data = flatten_for_update(batch, A)
        if extra is not None:
            data = tuple(np.concatenate([a, b]) for a, b in zip(data, flatten_for_update(extra, extra['adv'])))
        ppo_update(policy, opt, data, cfg, device)
        it += 1
        log.append(dict(arm=arm, seed=seed, iter=it, env_steps=env_steps,
                        train_success=float(batch['success'].mean()), **diag))
        if budget:
            if marks and env_steps >= marks[0]:
                while marks and env_steps >= marks[0]:
                    marks.pop(0)
                do_eval()
        elif it % cfg['eval_every'] == 0:
            do_eval()
    if log[-1].get('eval_success') is None:
        do_eval()
    return policy, log


# ---------------------------------------------------------------------------
# 9. Credit-accuracy study: do step advantages match ground-truth advantages?
# ---------------------------------------------------------------------------
@torch.no_grad()
def ground_truth_advantages(tasks, policy, batch, pairs, horizon, device, M=32):
    """A*(s_t, a_t) = Q(s_t, a_t) - sum_a pi(a|s_t) Q(s_t, a); Q = success probability
    within the remaining horizon from M brute-force rollouts per action."""
    tt = np.array([p[0] for p in pairs]); bb = np.array([p[1] for p in pairs])
    keys = batch['keys'][tt, bb]; tids = batch['task'][bb]
    n, nA = len(pairs), len(MOVES)
    rb = rollout(tasks, np.repeat(np.repeat(tids, nA), M), policy, horizon, device,
                 init_keys=np.repeat(np.repeat(keys, nA), M), first_action=np.repeat(np.tile(np.arange(nA), n), M),
                 keep_obs=False, init_t=np.repeat(np.repeat(tt, nA), M))
    Q = rb['success'].reshape(n, nA, M).mean(-1)
    pi = torch.softmax(policy(torch.as_tensor(batch['obs'][tt, bb], device=device)), -1).cpu().numpy()
    V = (pi * Q).sum(-1)
    return Q[np.arange(n), batch['actions'][tt, bb]] - V, Q, V


def credit_study(tasks, policy, cfg, device, estimators=('GRPO', 'GraphGPO', 'GraphGPO+FrontierMC', 'Progress+Tree'),
                 n_pairs=400, M=32, seed=0, tasks_per_iter=None):
    """All estimators' advantages on the SAME batch vs ground truth. Metrics:
    cosine (alignment of the advantage vector with the true one, a proxy for how
    well the policy-gradient direction is estimated), Spearman, sign agreement;
    split by whether the step's start state is linked to success in the graph."""
    from scipy.stats import spearmanr
    rng = np.random.default_rng(seed)
    P = tasks_per_iter or cfg['tasks_per_iter']
    tids = rng.choice(tasks.N, P, replace=False)
    group = np.repeat(np.arange(P), cfg['group_size'])
    batch = rollout(tasks, np.repeat(tids, cfg['group_size']), policy, cfg['horizon'], device,
                    merge_noise=cfg.get('merge_noise', 0.0))
    _, raw_d = graph_step_returns(batch, group, gamma=cfg['gamma'])
    tt, bb = np.nonzero(batch['active'])
    real_move = (batch['keys'][tt, bb] % SALT_BASE) != (batch['keys'][tt + 1, bb] % SALT_BASE)
    cand = np.where(real_move | (batch['keys'][tt + 1, bb] == SUCCESS))[0]
    pick = rng.choice(cand, min(n_pairs, len(cand)), replace=False)
    pairs = list(zip(tt[pick], bb[pick]))
    A_true, _, _ = ground_truth_advantages(tasks, policy, batch, pairs, cfg['horizon'], device, M)
    linked = np.array([raw_d[group[b]].get(int(batch['keys'][t, b]), math.inf) != math.inf for t, b in pairs])
    rows = []
    for est in estimators:
        A, _, cost, _ = compute_advantages(est, batch, group, tasks, policy, cfg, device, None, np.random.default_rng(seed))
        a_hat = np.array([A[t, b] for t, b in pairs])
        for name, mask in (('all', np.ones(len(pairs), bool)), ('linked', linked), ('frontier', ~linked)):
            cos = float(np.dot(a_hat[mask], A_true[mask]) / (np.linalg.norm(a_hat[mask]) * np.linalg.norm(A_true[mask]) + 1e-12)) if mask.sum() else np.nan
            m = mask & (np.abs(A_true) > 1e-9)
            if m.sum() < 5:
                rows.append(dict(estimator=est, subset=name, n=int(m.sum()), cosine=cos, spearman=np.nan,
                                 sign_agree=np.nan, extra_env_steps=cost))
                continue
            rho = spearmanr(a_hat[m], A_true[m]).correlation
            ms = m & (np.abs(A_true) > 0.05)
            sa = float(np.mean(np.sign(a_hat[ms]) == np.sign(A_true[ms]))) if ms.sum() else np.nan
            rows.append(dict(estimator=est, subset=name, n=int(m.sum()), cosine=cos, spearman=float(rho),
                             sign_agree=sa, extra_env_steps=cost))
    return rows, dict(batch_success=float(batch['success'].mean()), linked_frac=float(linked.mean()))

In [ ]:
import importlib, h2lab_core as H
importlib.reload(H)
import torch
torch.set_num_threads(1)

CFG = dict(H.DEFAULT_CFG)   # 9x9 map, 2 traps, 40-step horizon, G=8, same minibatch size (1,024 steps) for every arm
PRESETS = {
    'smoke': dict(budget=150_000, seeds=[0, 1], eval_points=5,
                  arms=['GraphGPO', 'Tree+GraphGPO', 'Progress+Tree', 'Progress+Tree(LOO)'],
                  cs_snapshots=[0, 15], cs_seeds=[0],
                  stress_noise=[0.9], stress_seeds=[0], stress_arms=['GraphGPO', 'Progress+Tree', 'Progress+Tree(LOO)'], stress_budget=100_000,
                  regime_seeds=[0], calib_noise=[0.5], sweep_noise=[0.9], sweep_dense=False,
                  sweep_variants=['P+T(LOO) spread32', 'P+T(critic) smart32']),
    'fast':  dict(budget=1_200_000, seeds=[0, 1, 2], eval_points=20,
                  arms=['GraphGPO', 'GraphGPO(2xG)', 'Tree+GRPO', 'Tree+GraphGPO', 'Progress+Tree', 'Progress+Tree(LOO)',
                        'Progress+Tree(random)'],
                  cs_snapshots=[0, 60, 120], cs_seeds=[0, 1],
                  stress_noise=[0.15, 0.5, 0.9], stress_seeds=[0, 1], stress_arms=['GraphGPO', 'Tree+GraphGPO', 'Progress+Tree', 'Progress+Tree(LOO)'],
                  stress_budget=1_200_000, regime_seeds=[0, 1, 2], calib_noise=[0.3, 0.7], sweep_noise=[0.15, 0.5], sweep_dense=True,
                  sweep_variants=['P+T(LOO) smart8', 'P+T(LOO) spread32', 'P+T(critic) smart32']),
    'full':  dict(budget=2_400_000, seeds=[0, 1, 2, 3, 4], eval_points=24,
                  arms=['GRPO', 'GraphGPO', 'GraphGPO(2xG)', 'GraphGPO+FrontierMC', 'Tree+GRPO', 'Tree+GraphGPO',
                        'Progress+Tree', 'Progress+Tree(LOO)', 'Progress+Tree(random)'],
                  cs_snapshots=[0, 60, 120, 200], cs_seeds=[0, 1, 2],
                  stress_noise=[0.15, 0.3, 0.5, 0.9], stress_seeds=[0, 1, 2], stress_arms=['GraphGPO', 'Tree+GraphGPO', 'Progress+Tree', 'Progress+Tree(LOO)'],
                  stress_budget=1_200_000, regime_seeds=[0, 1, 2, 3, 4], calib_noise=[0.1, 0.2, 0.4, 0.6, 0.7, 0.8],
                  sweep_noise=[0.15, 0.5], sweep_dense=True,
                  sweep_variants=['Progress(LOO)', 'P+T(LOO) smart8', 'P+T(LOO) spread32', 'P+T(critic) smart32']),
    # Step 1 of the plan, made stronger: the pre-registered comparison on NEW task pools (train/eval maps never
    # seen in v2 or v2.1) with 10 seeds. Everything else is skipped.
    'replicate': dict(budget=2_400_000, seeds=list(range(10)), eval_points=24,
                  arms=['GraphGPO', 'GraphGPO(2xG)', 'Tree+GraphGPO', 'Progress+Tree', 'Progress+Tree(LOO)'],
                  task_seed_train=3000, task_seed_eval=4000,
                  cs_snapshots=[], cs_seeds=[], stress_noise=[], stress_seeds=[], stress_arms=[], stress_budget=1_200_000,
                  regime_seeds=[], calib_noise=[], sweep_noise=[], sweep_dense=False, sweep_variants=[]),
}
P = PRESETS[MODE]
CFG.update(env_budget=P['budget'], eval_points=P['eval_points'],
           task_seed_train=P.get('task_seed_train', CFG['task_seed_train']), task_seed_eval=P.get('task_seed_eval', CFG['task_seed_eval']))
print(f"Arms: {P['arms']}\nSeeds: {P['seeds']} | env-step budget per run: {P['budget']:,}")
print(json.dumps(CFG, indent=1))

## 2. The environment
`KeyDoor-Traps`: pick up the correct key **K**, open the door **D**, reach the goal **G**. Picking up the wrong key **X** or stepping on a trap **^** is irreversible, but the episode continues to the horizon. This mimics a destructive tool call in an agentic task, and it fills failed trajectories with states that never connect to success.

In [ ]:
tasks = H.KeyDoorTasks(CFG['n_train_tasks'], CFG['size'], CFG['n_traps'], seed=1000)
def show(i):
    S = tasks.S; out = []
    for r in range(S):
        row = ''
        for c in range(S):
            ch = '#' if tasks.wall[i, r, c] else '.'
            if (r, c) == tuple(tasks.door[i]): ch = 'D'
            if tasks.trap[i, r, c]: ch = '^'
            for p, s in ((tasks.key[i], 'K'), (tasks.bad[i], 'X'), (tasks.goal[i], 'G'), (tasks.start[i], 'A')):
                if (r, c) == tuple(p): ch = s
            row += ch
        out.append(row)
    return '\n'.join(out)
print(show(0)); print(); print(show(1))
pol0 = H.MLPPolicy(tasks.obs_dim)
rb = H.rollout(tasks, np.repeat(np.arange(tasks.N), 8), pol0, CFG['horizon'], 'cpu', keep_obs=False)
_, _, inv, _, trapped = H.decode_key(np.clip(rb['keys'][-1], 0, None), tasks.S)
print(f"\nRandom-policy success: {rb['success'].mean():.3%} | ended trapped: {trapped.mean():.1%} | ended holding wrong key: {(inv == 2).mean():.1%}")

## 3. Main experiment: every arm runs to the same environment-step budget
`env_steps` counts **all** environment steps, including branches and MC continuations. Each (arm, seed) runs in its own process.

In [ ]:
def _job(args):
    import torch, warnings, h2lab_core as H
    warnings.filterwarnings('ignore')
    torch.set_num_threads(1)
    arm, seed, cfg, tag = args
    t = time.time()
    _, log = H.train_arm(arm, seed, cfg)
    for l in log:
        l['tag'] = tag
    return log, time.time() - t

ctx = mp.get_context('fork') if 'fork' in mp.get_all_start_methods() else mp.get_context()
def run_jobs(jobs, label):
    logs, t0 = [], time.time()
    with ctx.Pool(N_WORKERS) as pool:
        for i, (log, dt) in enumerate(pool.imap_unordered(_job, jobs)):
            logs += log
            last = [l for l in log if 'eval_success' in l][-1]
            print(f"[{label} {i+1}/{len(jobs)}] {log[0]['arm']:22s} seed {log[0]['seed']} {log[0]['tag']:>8s}  "
                  f"{dt:5.0f}s  iters {last['iter']:4d}  env {last['env_steps']:>9,}  final eval {last['eval_success']:.3f}")
    print(f"{label}: {time.time()-t0:.0f}s total")
    return pd.DataFrame(logs)

df = run_jobs([(a, s, CFG, 'main') for a in P['arms'] for s in P['seeds']], 'main')
ev = df[df.eval_success.notna()][['arm', 'seed', 'iter', 'env_steps', 'eval_success']].copy()
tr = df[df.train_success.notna()].copy()
ev.to_csv('h2lab_v2_eval.csv', index=False); tr.to_csv('h2lab_v2_train.csv', index=False)

### 3a. Kill test and branch diagnostics
`frontier_moving_step_frac`: share of moving steps whose start state has no path to success in GraphGPO's graph. If it stays below ~5%, there is little for any frontier method to fix. For the tree arms: branches per iteration, branch env steps, the share of candidate branch points skipped as absorbing or as cached dead ends, and the branch success rate.

In [ ]:
g = tr[tr.arm == 'GraphGPO'].groupby('iter')[['frontier_moving_step_frac', 'train_success']].mean()
ax = g.rolling(5, min_periods=1).mean().plot(figsize=(8, 3.2), title='GraphGPO: frontier share of moving steps vs training success')
ax.axhline(0.05, ls='--', c='gray'); ax.set_xlabel('iteration'); plt.show()
share = (g['frontier_moving_step_frac'] > 0.05).mean()
print(f"Frontier share > 5% in {share:.0%} of iterations -> {'PASS: continue' if share > 0.5 else 'KILL: little to fix here'}")
tree_arms = [a for a in P['arms'] if 'Tree' in a]
if tree_arms:
    bt = tr[tr.arm.isin(tree_arms)].groupby('arm')[['branch_nodes', 'branch_steps', 'branch_frozen_skipped', 'branch_dead_skipped', 'branch_success']].mean()
    main_steps = (tr[tr.arm.isin(tree_arms)].groupby('arm').env_steps.max() / tr[tr.arm.isin(tree_arms)].groupby('arm').iter.max())
    bt['branch share of env steps'] = bt.branch_steps / main_steps
    bt['skipped absorbing %'] = 100 * bt.branch_frozen_skipped / (bt.branch_nodes + bt.branch_frozen_skipped + bt.branch_dead_skipped)
    bt['skipped dead-end %'] = 100 * bt.branch_dead_skipped / (bt.branch_nodes + bt.branch_frozen_skipped + bt.branch_dead_skipped)
    display(bt.round(3))

### 3b. Learning curves
**Left (primary):** success vs environment steps. **Right:** success vs iterations. Arms with more samples per iteration also take more optimizer steps per iteration here; section 3e compares them with equal updates per iteration.

In [ ]:
def iqm(x):
    x = np.sort(np.asarray(x, float)); x = x[~np.isnan(x)]
    if len(x) == 0: return np.nan
    lo, hi = int(np.floor(0.25 * len(x))), int(np.ceil(0.75 * len(x)))
    return x[lo:hi].mean() if hi > lo else x.mean()

def boot_ci(x, stat=iqm, n=2000, seed=0):
    x = np.asarray(x, float); x = x[~np.isnan(x)]
    if len(x) < 2: return (np.nan, np.nan)
    rng = np.random.default_rng(seed)
    return tuple(np.percentile([stat(rng.choice(x, len(x))) for _ in range(n)], [2.5, 97.5]))

def at_budget(ev, xcol, budget):
    rows = []
    for (arm, seed), d in ev.groupby(['arm', 'seed']):
        d = d.sort_values(xcol)
        rows.append(dict(arm=arm, seed=seed, success=np.interp(budget, d[xcol], d.eval_success, right=np.nan)))
    return pd.DataFrame(rows)

def curves(ev, xcol, grid, ax, title, arms):
    for arm in arms:
        m, lo, hi = [], [], []
        for b in grid:
            v = at_budget(ev[ev.arm == arm], xcol, b).success.values
            m.append(iqm(v)); l, h = boot_ci(v); lo.append(l); hi.append(h)
        ax.plot(grid, m, label=arm, lw=2.2 if arm == 'Progress+Tree' else 1.4)
        ax.fill_between(grid, lo, hi, alpha=0.12)
    ax.set_title(title); ax.set_xlabel(xcol); ax.set_ylabel('eval success (IQM, 95% CI)')

B_steps = min(P['budget'], ev.groupby(['arm', 'seed']).env_steps.max().min())
B_iter = ev.groupby(['arm', 'seed']).iter.max().min()
fig, axs = plt.subplots(1, 2, figsize=(15, 4.8))
curves(ev, 'env_steps', np.linspace(0, B_steps, 25), axs[0], 'Success vs environment steps (incl. branches/probes)', P['arms'])
curves(ev, 'iter', np.linspace(0, B_iter, 25), axs[1], 'Success vs iterations', P['arms'])
axs[1].legend(fontsize=8); plt.tight_layout(); plt.show()

### 3c. Fixed-budget comparison and pass/fail
Pre-registered criterion: **≥ 3 points** better IQM success than the comparison arm with **non-overlapping 95% CIs**. The primary test is at equal environment steps against GraphGPO and against the budget-matched GraphGPO(2xG).

In [ ]:
def table(ev, xcol, budget, arms):
    d = at_budget(ev, xcol, budget)
    out = d.groupby('arm').success.agg(IQM=iqm, mean='mean', n='count')
    out['CI95'] = [tuple(np.round(boot_ci(d[d.arm == a].success.values), 3)) for a in out.index]
    return out.loc[[a for a in arms if a in out.index]]

def verdict(t, a, b):
    if a not in t.index or b not in t.index: return None
    diff = t.loc[a, 'IQM'] - t.loc[b, 'IQM']
    sep = t.loc[a, 'CI95'][0] > t.loc[b, 'CI95'][1]
    return diff, sep, ('PASS' if diff >= 0.03 and sep else 'not demonstrated')

t_steps = table(ev, 'env_steps', B_steps, P['arms']); t_iter = table(ev, 'iter', B_iter, P['arms'])
print(f"At equal ENVIRONMENT STEPS ({B_steps:,.0f}) — primary:"); display(t_steps.round(3))
print(f"At equal ITERATIONS ({B_iter}):"); display(t_iter.round(3))
FLAGSHIPS = [a for a in ['Progress+Tree', 'Progress+Tree(LOO)'] if a in P['arms']]
for f in FLAGSHIPS:
    for b in ['GraphGPO', 'GraphGPO(2xG)']:
        for name, t in (('env steps', t_steps), ('iterations', t_iter)):
            v = verdict(t, f, b)
            if v: print(f"{f:18s} vs {b:14s} at equal {name:10s}: {v[0]:+.3f} -> {v[2]} (CIs separated: {v[1]})")

In [ ]:
ITER_OVERHEAD = 2000   # fixed cost per iteration in env-step equivalents: weight sync, scheduling, waiting for the slowest rollout
ev['cost'] = ev.env_steps + ev.iter * ITER_OVERHEAD
B_cost = ev.groupby(['arm', 'seed']).cost.max().min()
fig, ax = plt.subplots(figsize=(8, 4))
curves(ev, 'cost', np.linspace(0, B_cost, 25), ax, f'Success vs total cost (ITER_OVERHEAD={ITER_OVERHEAD:,})', P['arms'])
ax.legend(fontsize=8); plt.show()
t_cost = table(ev, 'cost', B_cost, P['arms']); display(t_cost.round(3))
sweep = {}
for ov in [0, 1000, 5000, 20000]:
    c = ev.env_steps + ev.iter * ov
    e2 = ev.assign(cost=c); Bc = e2.groupby(['arm', 'seed']).cost.max().min()
    sweep[f'overhead={ov:,}'] = table(e2, 'cost', Bc, P['arms'])['IQM']
print('IQM success at equal total cost, by per-iteration overhead:'); display(pd.DataFrame(sweep).round(3))

### 3d. Attribution: more data, better credit, or better branch points?
At equal environment steps:
- **Data effect:** tree sampling with outcome-only credit (`Tree+GRPO`) or graph credit (`Tree+GraphGPO`) vs flat sampling.
- **Credit effect:** `Progress+Tree` vs `Tree+GraphGPO` and `Tree+GRPO` on the same kind of tree.
- **Branch-point effect:** `Progress+Tree` (smart) vs `Progress+Tree(random)` with the same number of branches.

In [ ]:
grid = pd.DataFrame(index=['GRPO credit', 'GraphGPO credit', 'Progress credit', 'Progress credit (LOO)'],
                    columns=['flat (G)', 'flat (2G)', 'tree (smart)', 'tree (random)'], dtype=float)
cellmap = {('GRPO credit', 'flat (G)'): 'GRPO', ('GRPO credit', 'flat (2G)'): 'GRPO(2xG)', ('GRPO credit', 'tree (smart)'): 'Tree+GRPO',
           ('GraphGPO credit', 'flat (G)'): 'GraphGPO', ('GraphGPO credit', 'flat (2G)'): 'GraphGPO(2xG)',
           ('GraphGPO credit', 'tree (smart)'): 'Tree+GraphGPO', ('Progress credit', 'tree (smart)'): 'Progress+Tree',
           ('Progress credit', 'tree (random)'): 'Progress+Tree(random)', ('Progress credit (LOO)', 'tree (smart)'): 'Progress+Tree(LOO)'}
for (r, c), arm in cellmap.items():
    if arm in t_steps.index: grid.loc[r, c] = t_steps.loc[arm, 'IQM']
print(f"IQM success at equal environment steps ({B_steps:,.0f}); empty = arm not run in this mode:")
display(grid.round(3))
pairs = [('Data effect, graph credit', 'Tree+GraphGPO', 'GraphGPO'), ('Data effect vs 2G, graph credit', 'Tree+GraphGPO', 'GraphGPO(2xG)'),
         ('Credit effect on tree: progress vs graph', 'Progress+Tree', 'Tree+GraphGPO'),
         ('Credit effect on tree: progress vs outcome', 'Progress+Tree', 'Tree+GRPO'),
         ('Branch-point effect: smart vs random', 'Progress+Tree', 'Progress+Tree(random)'),
         ('Baseline effect: leave-one-out vs telescoping', 'Progress+Tree(LOO)', 'Progress+Tree')]
rows = []
for name, a, b in pairs:
    v = verdict(t_steps, a, b)
    if v: rows.append(dict(effect=name, comparison=f'{a} − {b}', delta=v[0], CIs_separated=v[1]))
if rows: display(pd.DataFrame(rows).set_index('effect').round(3))

### 3e. Update-setting check: equal updates per iteration
Same budget, but every arm gets the same number of optimizer steps per iteration (`update_mode='fixed_steps'`), as in trainers that size minibatches per prompt (verl scales the PPO minibatch by the number of responses per prompt). Arms with more samples per iteration then take fewer, larger steps at equal env steps. **At equal iterations this isolates per-update credit quality; at equal env steps it shows how much the conclusion depends on the trainer's minibatching.**

In [ ]:
RUN_REGIME_CHECK = bool(P['regime_seeds'])
if RUN_REGIME_CHECK:
    RG_ARMS = [a for a in ['GraphGPO', 'Progress+Tree', 'Progress+Tree(LOO)'] if a in P['arms']]
    rg = run_jobs([(a, s, {**CFG, 'update_mode': 'fixed_steps'}, 'fixed_steps') for a in RG_ARMS
                   for s in P['regime_seeds']], 'regime')
    rge = rg[rg.eval_success.notna()][['arm', 'seed', 'iter', 'env_steps', 'eval_success']].copy()
    rB_steps = rge.groupby(['arm', 'seed']).env_steps.max().min(); rB_iter = rge.groupby(['arm', 'seed']).iter.max().min()
    rt_steps = table(rge, 'env_steps', rB_steps, RG_ARMS); rt_iter = table(rge, 'iter', rB_iter, RG_ARMS)
    print(f"fixed_steps — equal env steps ({rB_steps:,.0f}):"); display(rt_steps.round(3))
    print(f"fixed_steps — equal iterations ({rB_iter}):"); display(rt_iter.round(3))
    for f in RG_ARMS[1:]:
        for name, t in (('env steps', rt_steps), ('iterations', rt_iter)):
            v = verdict(t, f, 'GraphGPO')
            if v: print(f"fixed_steps: {f} vs GraphGPO at equal {name}: {v[0]:+.3f} -> {v[2]}")
    fig, axs = plt.subplots(1, 2, figsize=(13, 4))
    curves(rge, 'env_steps', np.linspace(0, rB_steps, 20), axs[0], 'fixed_steps: success vs env steps', RG_ARMS)
    curves(rge, 'iter', np.linspace(0, rB_iter, 20), axs[1], 'fixed_steps: success vs iterations', RG_ARMS)
    axs[1].legend(fontsize=8); plt.tight_layout(); plt.show()

## 4. Credit-accuracy study
Policy snapshots from a GraphGPO run; every estimator's advantages are computed on the **same batch** and compared with ground truth `A*(s,a) = Q(s,a) − Σπ(a'|s)Q(s,a')` (success probability within the remaining horizon, from 32 brute-force rollouts per action). **Cosine** = alignment of the advantage vector with the true one (how well the gradient direction is estimated); Spearman and sign agreement as in v1. Split into **linked** steps (start state has a path to success) and **frontier** steps. `+VineAll` estimators probe every visited state without training on the probes: they show each credit channel's ceiling. Frontier MC probes every group here (`mc_trigger='always'`).

In [ ]:
CS_EST = ['GRPO', 'GraphGPO', 'GraphGPO+FrontierMC', 'GraphGPO+VineAll', 'Progress', 'Progress+VineAll',
          'Tree+GRPO', 'Tree+GraphGPO', 'Progress+Tree', 'Progress+Tree(LOO)']
CS_CFG = {**CFG, 'mc_trigger': 'always', 'env_budget': None}

def _cs_job(args):
    import torch, warnings, h2lab_core as H
    warnings.filterwarnings('ignore')
    torch.set_num_threads(1)
    seed, snaps, cfg = args
    tasks = H.KeyDoorTasks(cfg['n_train_tasks'], cfg['size'], cfg['n_traps'], seed=1000)
    pol = H.MLPPolicy(tasks.obs_dim, hidden=cfg['hidden']); H.set_seed(seed)
    opt = torch.optim.Adam(pol.parameters(), lr=cfg['lr'])
    rng = np.random.default_rng(seed + 99); rows, prev = [], 0
    for snap in snaps:
        for it in range(prev, snap):  # continue one GraphGPO run up to the snapshot
            tids = rng.choice(tasks.N, cfg['tasks_per_iter'], replace=False)
            grp = np.repeat(np.arange(cfg['tasks_per_iter']), cfg['group_size'])
            b = H.rollout(tasks, np.repeat(tids, cfg['group_size']), pol, cfg['horizon'], 'cpu')
            A, _, _, _ = H.compute_advantages('GraphGPO', b, grp, tasks, pol, cfg, 'cpu')
            H.ppo_update(pol, opt, H.flatten_for_update(b, A), cfg, 'cpu')
        prev = snap
        r, info = H.credit_study(tasks, pol, cfg, 'cpu', estimators=CS_EST, n_pairs=400, M=32,
                                 seed=seed * 1000 + snap, tasks_per_iter=32)
        for x in r: x.update(seed=seed, snapshot=snap, **info)
        rows += r
    return rows

RUN_CREDIT_STUDY = bool(P['cs_seeds'])
if RUN_CREDIT_STUDY:
    with ctx.Pool(N_WORKERS) as pool:
        cs = pd.DataFrame([r for rows in pool.map(_cs_job, [(s, P['cs_snapshots'], CS_CFG) for s in P['cs_seeds']]) for r in rows])
    cs.to_csv('h2lab_v2_credit.csv', index=False)
    late = cs[cs.snapshot > 0] if (cs.snapshot > 0).any() else cs
    summ = late.groupby(['estimator', 'subset'])[['cosine', 'spearman', 'sign_agree']].mean().unstack('subset')
    cost = late[late.subset == 'all'].groupby('estimator').extra_env_steps.mean().rename('extra env steps / batch')
    ctab = pd.concat([summ[('cosine', 'all')].rename('cosine all'), summ[('cosine', 'frontier')].rename('cosine frontier'),
                      summ[('cosine', 'linked')].rename('cosine linked'), summ[('spearman', 'frontier')].rename('Spearman frontier'),
                      summ[('sign_agree', 'frontier')].rename('sign-agree frontier'), cost], axis=1).loc[CS_EST]
    print(f"Averaged over snapshots {sorted(late.snapshot.unique().tolist())} and seeds {P['cs_seeds']}:"); display(ctab.round(3))
    fig, ax = plt.subplots(figsize=(11, 3.8))
    ctab[['cosine all', 'cosine frontier', 'cosine linked']].plot.bar(ax=ax, rot=25, title='Gradient alignment with ground truth (cosine)')
    ax.axhline(0, c='k', lw=0.5); plt.tight_layout(); plt.show()

## 5. Sparse-graph stress test
In LLM agents, observations rarely repeat exactly, so identical-state merging (GraphGPO's graph, value pooling) has much less to work with. Here each visited state gets a unique graph identity with probability `merge_noise` (the environment itself is unchanged). The `merge_noise = 0` column comes from the main experiment; `fast` adds noise 0.15 and `full` adds 0.15 and 0.3, to find where v2 stops winning. **Question:** how much do v2's gains depend on identical states merging? Pure progress credit puts a trajectory's whole advantage on the steps where it shares states with others, so it needs merging; the leave-one-out variant falls back to RLOO-style credit where nothing merges.

In [ ]:
RUN_STRESS = bool(P['stress_noise'])
if RUN_STRESS:
    jobs = [(a, s, {**CFG, 'env_budget': P['stress_budget'], 'merge_noise': q}, f'noise={q}')
            for q in P['stress_noise'] for a in P['stress_arms'] for s in P['stress_seeds']]
    sdf = run_jobs(jobs, 'stress')
    sev = sdf[sdf.eval_success.notna()][['arm', 'seed', 'iter', 'env_steps', 'eval_success', 'tag']].copy()
    rows = []
    for arm in P['stress_arms']:
        if arm in ev.arm.unique():
            v = at_budget(ev[ev.arm == arm], 'env_steps', P['stress_budget']).success.values
            rows.append(dict(arm=arm, merge_noise=0.0, IQM=iqm(v), CI95=tuple(np.round(boot_ci(v), 3)), seeds=len(v)))
        for q in P['stress_noise']:
            d = sev[(sev.arm == arm) & (sev.tag == f'noise={q}')]
            v = at_budget(d, 'env_steps', P['stress_budget']).success.values
            rows.append(dict(arm=arm, merge_noise=q, IQM=iqm(v), CI95=tuple(np.round(boot_ci(v), 3)), seeds=len(v)))
    st = pd.DataFrame(rows)
    st.to_csv('h2lab_v2_stress.csv', index=False)
    display(st.pivot(index='arm', columns='merge_noise', values='IQM').round(3))
    fig, ax = plt.subplots(figsize=(6.5, 3.8))
    for arm, d in st.groupby('arm'):
        ax.plot(d.merge_noise, d.IQM, marker='o', label=arm)
    ax.set_xlabel('merge_noise (share of states that never merge)'); ax.set_ylabel(f"IQM success at {P['stress_budget']:,} env steps")
    ax.set_title('Sparse-graph stress test'); ax.legend(fontsize=8); plt.show()
    fr = sdf[sdf.train_success.notna() & (sdf.arm == 'GraphGPO')].groupby('tag').frontier_moving_step_frac.mean()
    print('GraphGPO frontier share of moving steps by merge noise:', fr.round(3).to_dict(),
          '| noise=0:', round(tr[tr.arm == 'GraphGPO'].frontier_moving_step_frac.mean(), 3))

### 5b. Where do real agent environments sit? Merge-rate calibration
The toy's `merge_noise` is a knob; real environments have a merge rate. The common statistic: the share of steps (after the first) whose state was also visited by another rollout of the same group of 8. Every iteration's main batch records it (`shared_step_frac`). Below: the toy at each noise level, by training success, with **ALFWorld measured** under GraphGPO's own state representation (observation + location/holding + moved items + admissible commands; `port/ProgressTree/tools/alfworld_merge_probe.py`, 16 games × 8 rollouts, uniform-random and partly-expert policies). Exact merging, and with GraphGPO's similarity merging (`enable_similarity`, difflib ratio ≥ 0.95).

In [ ]:
ALFWORLD = pd.DataFrame([
    dict(policy='uniform random', success=0.00, exact=0.44, similarity=0.68),
    dict(policy='50% expert',     success=0.61, exact=0.59, similarity=0.77),
    dict(policy='80% expert',     success=0.75, exact=0.71, similarity=0.79),
    dict(policy='95% expert',     success=0.83, exact=0.91, similarity=0.93)])
frames = []
if 'tr' in globals():
    frames.append(tr.assign(noise=0.0))                     # every arm's main batches (the statistic doesn't depend on the credit rule)
if 'sdf' in globals():
    d = sdf[sdf.train_success.notna()]
    frames.append(d.assign(noise=d.tag.str.replace('noise=', '').astype(float)))
if P['calib_noise']:
    cdf = run_jobs([('GraphGPO', 0, {**CFG, 'env_budget': P['stress_budget'], 'merge_noise': q}, f'noise={q}') for q in P['calib_noise']], 'calibration')
    d = cdf[cdf.train_success.notna()]
    frames.append(d.assign(noise=d.tag.str.replace('noise=', '').astype(float)))
RUN_CALIB = len(frames) > 0 and pd.concat(frames).noise.nunique() > 1
if RUN_CALIB:
    cal = pd.concat(frames)
    edges = [0, 0.1, 0.25, 0.4, 0.55, 0.7, 1.0]
    cal['success_bin'] = pd.cut(cal.train_success, edges, include_lowest=True)
    curve = cal.groupby(['success_bin', 'noise'], observed=False).shared_step_frac.mean().unstack('noise')
    cal[['noise', 'train_success', 'shared_step_frac']].to_csv('h2lab_v2_calib.csv', index=False)
    print('Toy (main batches of every arm): share of steps on a state another rollout of the group also visited, by training success and merge_noise')
    display(curve.round(2))

    # The toy's shared share barely depends on training success at a given noise level (rows of the table above),
    # so map ALFWorld's numbers through the pooled curve: mean shared share per noise level over all iterations.
    pooled = cal.groupby('noise').shared_step_frac.mean()
    print('pooled:', pooled.round(3).to_dict())

    def equivalent_noise(success, shared):
        xs, ys = pooled.index.values.astype(float), pooled.values     # ys falls as noise rises
        if shared >= ys.max(): return 0.0                             # as dense as the toy without noise (or denser)
        return float(np.interp(shared, ys[::-1], xs[::-1]))

    ALFWORLD['toy noise ≈ (exact)'] = [equivalent_noise(a, b) for a, b in zip(ALFWORLD.success, ALFWORLD.exact)]
    ALFWORLD['toy noise ≈ (similarity)'] = [equivalent_noise(a, b) for a, b in zip(ALFWORLD.success, ALFWORLD.similarity)]
    display(ALFWORLD.round(2))
    fig, ax = plt.subplots(figsize=(7, 4))
    mids = [(edges[i] + edges[i + 1]) / 2 for i in range(len(edges) - 1)]
    for q in curve.columns:
        y = curve[q].values
        ok = ~np.isnan(y)
        ax.plot(np.array(mids)[ok], y[ok], marker='.', label=f'toy, noise {q}', alpha=0.8)
    ax.scatter(ALFWORLD.success, ALFWORLD.exact, c='k', marker='o', s=60, label='ALFWorld, exact', zorder=5)
    ax.scatter(ALFWORLD.success, ALFWORLD.similarity, c='k', marker='s', s=60, facecolors='none', label='ALFWorld, similarity 0.95', zorder=5)
    ax.set_xlabel('training success'); ax.set_ylabel('share of steps on a shared state'); ax.set_ylim(0, 1)
    ax.legend(fontsize=7, ncol=2); ax.set_title('Merge-rate calibration'); plt.tight_layout(); plt.show()
else:
    print('Calibration needs runs at two or more noise levels (main experiment + stress test, or calib_noise).')

## 6. Ablations (optional)
Knobs of the v2 concept, each run as `Progress+Tree` at the main budget for the listed seeds. Compare against the main table at equal environment steps.

In [ ]:
RUN_ABLATIONS = False
ABLATIONS = {
    'branch_max_nodes=16':      dict(branch_max_nodes=16),
    'branch_max_nodes=64':      dict(branch_max_nodes=64),
    'branch_K=2':               dict(branch_K=2),
    'no dead-end cache':        dict(dead_min_n=10**9),
    'sigma_min=0.5':            dict(progress_sigma_min=0.5),
}
if RUN_ABLATIONS:
    ab = run_jobs([('Progress+Tree', s, {**CFG, **kw}, tag) for tag, kw in ABLATIONS.items() for s in P['seeds']], 'ablations')
    abe = ab[ab.eval_success.notna()]
    B_ab = min(B_steps, abe.groupby(['tag', 'seed']).env_steps.max().min())
    rows = []
    for tag, d in abe.groupby('tag'):
        v = at_budget(d, 'env_steps', B_ab).success.values
        rows.append(dict(ablation=tag, IQM=iqm(v), CI95=tuple(np.round(boot_ci(v), 3))))
    print(f'At equal environment steps ({B_ab:,.0f}):')
    display(pd.DataFrame(rows).set_index('ablation').round(3)); display(table(ev, 'env_steps', B_ab, P['arms']).round(3))

## 6b. Sparse-graph remedies and branch budget
Where states rarely merge, each branch informs only its own branch point and pooled values rest on one or two samples. Candidate remedies, each run at the stress test's budget and seeds, at noise 0.15 (where ALFWorld sits with similarity merging, section 5b) and 0.5 (and at noise 0 to check they cost nothing when states do merge):
- `spread32`: branch points spread evenly along each trajectory (instead of the most-visited states), so every segment gets a comparison.
- `critic`: a small learned value function V(obs) replaces pooled Monte Carlo values where fewer than 2 continuations passed through a state (progress credit still telescopes).
- `smart8`, `smart8 K4`: a quarter of the branch points (8 per group instead of 32), with 1 or 4 continuations each.
- `Progress(LOO)`: no tree at all, only the credit change (the cheapest port).
Reference rows (`GraphGPO`, `Tree+GraphGPO`, `P+T(LOO) smart32` = `Progress+Tree(LOO)`) come from sections 3 and 5.

In [ ]:
VARIANTS = {
    'Progress(LOO)':        ('Progress(LOO)', {}),
    'P+T(LOO) smart8':      ('Progress+Tree(LOO)', {'branch_max_nodes': 8}),
    'P+T(LOO) smart8 K4':   ('Progress+Tree(LOO)', {'branch_max_nodes': 8, 'branch_K': 4}),
    'P+T(LOO) spread32':    ('Progress+Tree(LOO,spread)', {}),
    'P+T(critic) smart32':  ('Progress+Tree(critic)', {}),
    'P+T(critic) spread32': ('Progress+Tree(critic,spread)', {}),
}
REFS = {'GraphGPO': 'GraphGPO', 'Tree+GraphGPO': 'Tree+GraphGPO', 'P+T(LOO) smart32': 'Progress+Tree(LOO)'}
RUN_SWEEP = bool(P['sweep_noise']) and bool(P['sweep_variants'])
if RUN_SWEEP:
    noises = P['sweep_noise'] + ([0.0] if P['sweep_dense'] else [])
    jobs = [(VARIANTS[t][0], s, {**CFG, 'env_budget': P['stress_budget'], 'merge_noise': q, **VARIANTS[t][1]}, f'{t}|{q}')
            for q in noises for t in P['sweep_variants'] for s in P['stress_seeds']]
    swdf = run_jobs(jobs, 'sweep')
    swe = swdf[swdf.eval_success.notna()].copy()
    swe['variant'] = swe.tag.str.split('|').str[0]; swe['noise'] = swe.tag.str.split('|').str[1].astype(float)
    B_sw = P['stress_budget']
    vals = {}                                                         # (variant, noise) -> per-seed success at budget
    for (t, q), d in swe.groupby(['variant', 'noise']):
        vals[(t, q)] = at_budget(d, 'env_steps', B_sw).success.values
    for name, arm in REFS.items():
        for q in noises:
            if q == 0.0 and 'ev' in globals():
                d = ev[(ev.arm == arm) & ev.seed.isin(P['stress_seeds'])]
            elif 'sev' in globals():
                d = sev[(sev.arm == arm) & (sev.tag == f'noise={q}')]
            else:
                continue
            if len(d): vals[(name, q)] = at_budget(d, 'env_steps', B_sw).success.values
    rows = [dict(variant=t, noise=q, IQM=iqm(v), CI95=tuple(np.round(boot_ci(v), 3)), seeds=len(v)) for (t, q), v in vals.items()]
    sw_tab = pd.DataFrame(rows)
    sw_tab.to_csv('h2lab_v2_sweep.csv', index=False)
    pv_sw = sw_tab.pivot(index='variant', columns='noise', values='IQM')
    order = [r for r in list(REFS) + list(VARIANTS) if r in pv_sw.index]
    print(f'IQM eval success at {B_sw:,} env steps:'); display(pv_sw.loc[order].round(3))
    def sw_verdict(a, b, q):
        if (a, q) not in vals or (b, q) not in vals: return None
        va, vb = vals[(a, q)], vals[(b, q)]
        diff = iqm(va) - iqm(vb); sep = boot_ci(va)[0] > boot_ci(vb)[1]
        return diff, sep, ('PASS' if diff >= 0.03 and sep else 'not demonstrated')
    vrows = []
    for t in [v for v in VARIANTS if v in pv_sw.index]:
        for q in noises:
            for ref in ('GraphGPO', 'P+T(LOO) smart32'):
                v = sw_verdict(t, ref, q)
                if v: vrows.append(dict(variant=t, noise=q, vs=ref, delta=round(v[0], 3), CIs_separated=v[1], verdict=v[2]))
    sw_ver = pd.DataFrame(vrows)
    if len(sw_ver): display(sw_ver.set_index(['variant', 'noise', 'vs']))

## 7. Optional LLM track (GPU)
The same environment rendered as ASCII text with **Qwen2.5 + LoRA** as the policy. The model scores the four action words at the last position (exact log-probabilities for PPO; no full-vocabulary projection). Multi-seed comparison at an equal environment-step budget, with a timing probe that projects the run time before the full runs and skips them if it exceeds `LLM_MAX_HOURS`. Results are checkpointed to `h2lab_v2_llm.csv` after every run, and an LLM section is appended to the summary.

**On Colab's G4 runtime** (NVIDIA RTX PRO 6000 Blackwell, ~96 GB, bf16): run the GPU check, set `RUN_LLM_TRACK = True`, and start with Qwen2.5-0.5B and 3 seeds; move to 1.5B once the timing probe looks affordable. `LLM_MERGE_NOISE = 0.5` repeats the sparse-graph stress test at LLM scale.

In [ ]:
import os, torch
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available(), '| CPUs:', os.cpu_count())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name} | {p.total_memory / 2**30:.0f} GiB | bf16: {torch.cuda.is_bf16_supported()}")

In [ ]:
%%writefile h2lab_llm.py
# ===== Optional LLM track: same environment rendered as text, LLM + LoRA policy =====
# The policy scores the four action words after a chat prompt (constrained
# action decoding). It exposes the same interface as MLPPolicy, so the rollout,
# graph, frontier-MC and PPO code above is reused unchanged.
import numpy as np
import torch
import torch.nn as nn

LEGEND = ("Legend: '#' wall, 'A' you, 'K' the correct key, 'X' a wrong key "
          "(picking it up makes the task impossible), '^' trap (freezes you), "
          "'D' locked door (walk into it while holding K), 'G' goal, '.' floor.")


def render_obs_text(o, S):
    g = o[:7 * S * S].reshape(7, S, S)
    inv = int(np.argmax(o[7 * S * S:7 * S * S + 3])); trapped = o[-1] > 0.5
    rows = []
    for r in range(S):
        line = ''
        for c in range(S):
            ch = '.'
            if g[0, r, c]: ch = '#'
            if g[1, r, c]: ch = '^'
            if g[3, r, c]: ch = 'D'
            if g[2, r, c]: ch = 'G'
            if g[4, r, c]: ch = 'K'
            if g[5, r, c]: ch = 'X'
            if g[6, r, c]: ch = 'A'
            line += ch
        rows.append(line)
    holding = ['nothing', 'the correct key K', 'the wrong key X'][inv]
    status = ' You are frozen in a trap.' if trapped else ''
    return '\n'.join(rows) + f"\nYou are holding {holding}.{status}"


class LLMPolicy(nn.Module):
    def __init__(self, model, tokenizer, S, action_words=('up', 'down', 'left', 'right'),
                 infer_batch=64):
        super().__init__()
        self.model, self.tok, self.S = model, tokenizer, S
        self.infer_batch = infer_batch
        self.tok.padding_side = 'left'
        if self.tok.pad_token is None:
            self.tok.pad_token = self.tok.eos_token
        ids = [self._first_token(w) for w in action_words]
        if len(set(ids)) < len(ids):  # fall back to single letters if words collide
            action_words = ('U', 'D', 'L', 'R'); ids = [self._first_token(w) for w in action_words]
        assert len(set(ids)) == len(ids), "action tokens must be distinct"
        self.action_words = action_words
        self.register_buffer('action_ids', torch.tensor(ids), persistent=False)

    def _first_token(self, w):
        return self.tok.encode(w, add_special_tokens=False)[0]

    def prompt(self, o):
        user = (f"You control agent A in a grid world. Pick up the correct key K, open the door D, "
                f"then reach the goal G.\n{LEGEND}\n\n{render_obs_text(o, self.S)}\n\n"
                f"Reply with exactly one word: {', '.join(self.action_words)}.")
        if getattr(self.tok, 'chat_template', None):
            return self.tok.apply_chat_template([{'role': 'user', 'content': user}],
                                                tokenize=False, add_generation_prompt=True)
        return user + "\nAnswer: "

    def forward(self, obs):
        """Logits of the 4 action words at the last position only (no full-vocabulary projection)."""
        obs_np = obs.detach().float().cpu().numpy() if torch.is_tensor(obs) else obs
        enc = self.tok([self.prompt(o) for o in obs_np], return_tensors='pt', padding=True).to(self.action_ids.device)
        base = self.model.get_base_model() if hasattr(self.model, 'get_base_model') else self.model
        h = base.model(input_ids=enc['input_ids'], attention_mask=enc['attention_mask']).last_hidden_state[:, -1, :]
        logits = h @ base.lm_head.weight[self.action_ids].T
        if getattr(base.lm_head, 'bias', None) is not None:
            logits = logits + base.lm_head.bias[self.action_ids]
        return logits.float()

    @torch.no_grad()
    def act(self, obs_np, device):
        logits = torch.cat([self.forward(obs_np[i:i + self.infer_batch])
                            for i in range(0, len(obs_np), self.infer_batch)])
        dist = torch.distributions.Categorical(logits=logits)
        a = dist.sample()
        return a.cpu().numpy(), dist.log_prob(a).cpu().numpy()


def _disable_incompatible_torchao():
    """Colab preinstalls an old torchao (e.g. 0.10). Recent PEFT raises ImportError
    from is_torchao_available() when it finds torchao < 0.16, even for plain,
    unquantized LoRA. We never use torchao, so if the installed version is too old,
    make PEFT treat it as absent. Works in a running session (no restart needed)."""
    import sys, importlib.metadata as md
    from packaging import version
    try:
        v = version.parse(md.version('torchao'))
    except md.PackageNotFoundError:
        return
    if v >= version.parse('0.16.0'):
        return
    import peft, peft.import_utils  # noqa: F401
    try:
        import peft.tuners.lora.torchao, peft.utils.quantization_utils  # noqa: F401
    except Exception:
        pass
    absent = lambda: False
    for name, mod in list(sys.modules.items()):
        if name.startswith('peft') and hasattr(mod, 'is_torchao_available'):
            setattr(mod, 'is_torchao_available', absent)
    print(f"[h2lab] torchao {v} is too old for this PEFT version; ignoring it (not needed for LoRA).")


def load_llm_policy(model_name, S, device, lora_r=16, infer_batch=256):
    from transformers import AutoModelForCausalLM, AutoTokenizer
    _disable_incompatible_torchao()
    from peft import LoraConfig, get_peft_model
    tok = AutoTokenizer.from_pretrained(model_name)
    use_bf16 = device == 'cuda' and torch.cuda.is_bf16_supported()
    dt = torch.bfloat16 if use_bf16 else torch.float32
    try:
        model = AutoModelForCausalLM.from_pretrained(model_name, dtype=dt)        # transformers >= 4.56
    except TypeError:
        model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=dt)  # older versions
    model = get_peft_model(model, LoraConfig(r=lora_r, lora_alpha=2 * lora_r, lora_dropout=0.0,
                                             target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj']))
    return LLMPolicy(model, tok, S, infer_batch=infer_batch).to(device)

In [ ]:
# ===== Section 7 (G4 version): LLM track — multi-seed, equal env-step budgets =====
RUN_LLM_TRACK = False   # set True on a GPU runtime (G4 recommended)
LLM_NAME = 'Qwen/Qwen2.5-0.5B-Instruct'      # start here; then try 'Qwen/Qwen2.5-1.5B-Instruct'
LLM_ARMS = ['GraphGPO', 'Progress+Tree(LOO)']  # add 'Tree+GraphGPO' to separate the data effect from the credit effect
LLM_SEEDS = [0, 1, 2]
LLM_BUDGET = 100_000       # environment steps per run, the same for every arm
LLM_MERGE_NOISE = 0.0      # 0.5 = sparse-graph stress test at LLM scale
LLM_INFER_BATCH = 256      # prompts per forward pass during rollouts
LLM_MINIBATCH = 64         # samples per PPO minibatch, the same for every arm
LLM_MAX_HOURS = 3.0        # skip the full runs if the timing probe projects more than this

if RUN_LLM_TRACK:
    import gc, time, importlib, torch, h2lab_llm as L
    from IPython.display import Markdown, display
    importlib.reload(L)
    dev = 'cuda' if torch.cuda.is_available() else 'cpu'
    assert dev == 'cuda' or os.environ.get('H2LAB_ALLOW_CPU'), 'The LLM track needs a GPU runtime (Runtime > Change runtime type > G4).'
    torch.set_num_threads(min(8, os.cpu_count() or 1))

    # Speed patch: the policy only needs the 4 action-word logits at the last position, so skip the
    # full-vocabulary projection at every position (the largest cost in the original forward pass).
    def _fast_forward(self, obs):
        obs_np = obs.detach().float().cpu().numpy() if torch.is_tensor(obs) else obs
        enc = self.tok([self.prompt(o) for o in obs_np], return_tensors='pt', padding=True).to(self.action_ids.device)
        base = self.model.get_base_model() if hasattr(self.model, 'get_base_model') else self.model
        h = base.model(input_ids=enc['input_ids'], attention_mask=enc['attention_mask']).last_hidden_state[:, -1, :]
        logits = h @ base.lm_head.weight[self.action_ids].T
        if getattr(base.lm_head, 'bias', None) is not None:
            logits = logits + base.lm_head.bias[self.action_ids]
        return logits.float()
    L.LLMPolicy.forward = _fast_forward

    LCFG = dict(H.DEFAULT_CFG)
    LCFG.update(size=7, n_traps=1, horizon=20, tasks_per_iter=16, group_size=8, n_train_tasks=128, n_eval_tasks=32,
                env_budget=LLM_BUDGET, max_iters=1000, eval_points=8, lr=1e-4, ppo_epochs=1,
                update_mode='fixed_minibatch', minibatch=LLM_MINIBATCH, mc_horizon=10, branch_max_nodes=16,
                merge_noise=LLM_MERGE_NOISE)
    ltrain = H.KeyDoorTasks(LCFG['n_train_tasks'], LCFG['size'], LCFG['n_traps'], seed=1000)
    leval = H.KeyDoorTasks(LCFG['n_eval_tasks'], LCFG['size'], LCFG['n_traps'], seed=2000)
    make_policy = lambda: L.load_llm_policy(LLM_NAME, LCFG['size'], dev, infer_batch=LLM_INFER_BATCH)

    # Helpers (already defined if sections 3b-3c ran in this session)
    if 'table' not in globals():
        def iqm(x):
            x = np.sort(np.asarray(x, float)); x = x[~np.isnan(x)]
            if len(x) == 0: return np.nan
            lo, hi = int(np.floor(0.25 * len(x))), int(np.ceil(0.75 * len(x)))
            return x[lo:hi].mean() if hi > lo else x.mean()
        def boot_ci(x, stat=iqm, n=2000, seed=0):
            x = np.asarray(x, float); x = x[~np.isnan(x)]
            if len(x) < 2: return (np.nan, np.nan)
            rng = np.random.default_rng(seed)
            return tuple(np.percentile([stat(rng.choice(x, len(x))) for _ in range(n)], [2.5, 97.5]))
        def at_budget(ev, xcol, budget):
            return pd.DataFrame([dict(arm=a, seed=s, success=np.interp(budget, d.sort_values(xcol)[xcol], d.sort_values(xcol).eval_success, right=np.nan))
                                 for (a, s), d in ev.groupby(['arm', 'seed'])])
        def table(ev, xcol, budget, arms):
            d = at_budget(ev, xcol, budget)
            out = d.groupby('arm').success.agg(IQM=iqm, mean='mean', n='count')
            out['CI95'] = [tuple(np.round(boot_ci(d[d.arm == a].success.values), 3)) for a in out.index]
            return out.loc[[a for a in arms if a in out.index]]
        def verdict(t, a, b):
            if a not in t.index or b not in t.index: return None
            diff = t.loc[a, 'IQM'] - t.loc[b, 'IQM']; sep = t.loc[a, 'CI95'][0] > t.loc[b, 'CI95'][1]
            return diff, sep, ('PASS' if diff >= 0.03 and sep else 'not demonstrated')

    # 1) Prompt check and the base model's starting success
    pol = make_policy()
    print(pol.prompt(H.VecEnv(ltrain, [0], LCFG['horizon']).obs()[0])); print('action words:', pol.action_words)
    base_success = H.evaluate(leval, np.arange(leval.N), pol, LCFG['horizon'], dev)
    print(f"Base model success before RL: {base_success:.3f}")

    # 2) Timing probe: 2 iterations per arm -> projected run time
    def _sec_per_step(arm, pol, n=2):
        opt = torch.optim.Adam([p for p in pol.parameters() if p.requires_grad], lr=LCFG['lr'])
        rng = np.random.default_rng(0); mem = dict(cache={}, start={}); steps = 0; t0 = time.time()
        for _ in range(n):
            G = LCFG['group_size'] * H.arm_spec(arm).get('gmult', 1)
            tids = rng.choice(ltrain.N, LCFG['tasks_per_iter'], replace=False)
            grp = np.repeat(np.arange(LCFG['tasks_per_iter']), G)
            b = H.rollout(ltrain, np.repeat(tids, G), pol, LCFG['horizon'], dev, merge_noise=LCFG['merge_noise'])
            A, extra, cost, _ = H.compute_advantages(arm, b, grp, ltrain, pol, LCFG, dev, mem, rng)
            data = H.flatten_for_update(b, A)
            if extra is not None:
                data = tuple(np.concatenate([x, y]) for x, y in zip(data, H.flatten_for_update(extra, extra['adv'])))
            H.ppo_update(pol, opt, data, LCFG, dev)
            steps += b['env_steps'] + cost
        if dev == 'cuda': torch.cuda.synchronize()
        return (time.time() - t0) / steps, steps / n
    proj = {}
    for arm in LLM_ARMS:
        sps, spi = _sec_per_step(arm, pol)
        proj[arm] = sps * LLM_BUDGET * 1.15 / 60   # +15% for evaluation and model loading
        print(f"{arm:22s} {sps*1000:6.1f} ms per env step, {spi:,.0f} env steps per iteration -> ~{proj[arm]:.0f} min per run")
    del pol; gc.collect()
    if dev == 'cuda': torch.cuda.empty_cache()
    total_h = sum(proj.values()) * len(LLM_SEEDS) / 60
    print(f"Projected total: {total_h:.1f} h for {len(LLM_ARMS)} arms x {len(LLM_SEEDS)} seeds")

    # 3) Full runs (results are checkpointed to h2lab_v2_llm.csv after every run)
    if total_h > LLM_MAX_HOURS:
        print(f"Skipping the full runs: projected {total_h:.1f} h > LLM_MAX_HOURS = {LLM_MAX_HOURS}. "
              "Lower LLM_BUDGET, use fewer seeds or a smaller model, or raise LLM_MAX_HOURS.")
    else:
        llm_logs = []
        for seed in LLM_SEEDS:
            for arm in LLM_ARMS:
                t0 = time.time()
                _, log = H.train_arm(arm, seed, LCFG, dev, train_tasks=ltrain, eval_tasks=leval, policy_factory=make_policy)
                for l in log:
                    l.update(model=LLM_NAME, merge_noise=LLM_MERGE_NOISE)
                llm_logs += log
                pd.DataFrame(llm_logs).to_csv('h2lab_v2_llm.csv', index=False)
                last = [l for l in log if 'eval_success' in l][-1]
                print(f"[{arm} | seed {seed}] {(time.time()-t0)/60:5.1f} min  iters {last['iter']:4d}  "
                      f"env {last['env_steps']:>8,}  final eval {last['eval_success']:.3f}")
                gc.collect()
                if dev == 'cuda': torch.cuda.empty_cache()

        # 4) Results at equal environment steps and at equal iterations
        llm_ev = pd.DataFrame(llm_logs); llm_ev = llm_ev[llm_ev.eval_success.notna()]
        LB = min(LLM_BUDGET, llm_ev.groupby(['arm', 'seed']).env_steps.max().min())
        LBi = llm_ev.groupby(['arm', 'seed']).iter.max().min()
        lt_steps = table(llm_ev, 'env_steps', LB, LLM_ARMS); lt_iter = table(llm_ev, 'iter', LBi, LLM_ARMS)
        print(f"Equal environment steps ({LB:,.0f}):"); display(lt_steps.round(3))
        print(f"Equal iterations ({LBi}):"); display(lt_iter.round(3))
        fig, axs = plt.subplots(1, 2, figsize=(13, 4))
        for arm, d in llm_ev.groupby('arm'):
            for s, ds in d.groupby('seed'):
                axs[0].plot(ds.env_steps, ds.eval_success, color=f"C{LLM_ARMS.index(arm)}", alpha=0.6, label=arm if s == LLM_SEEDS[0] else None)
                axs[1].plot(ds.iter, ds.eval_success, color=f"C{LLM_ARMS.index(arm)}", alpha=0.6)
        axs[0].set_xlabel('env steps'); axs[1].set_xlabel('iteration'); axs[0].set_ylabel('eval success'); axs[0].legend(); plt.show()

        # 5) Append an LLM section to the findings summary
        lines = [f"## LLM track: {LLM_NAME} + LoRA ({len(LLM_SEEDS)} seeds, {LB:,.0f} env steps, merge noise {LLM_MERGE_NOISE})",
                 f"- Base model success before RL: {base_success:.3f}."]
        for a in LLM_ARMS:
            if a in lt_steps.index:
                lo, hi = lt_steps.loc[a, 'CI95']
                lines.append(f"- `{a}`: IQM {lt_steps.loc[a, 'IQM']:.3f} (CI {lo:.3f}–{hi:.3f}) at equal env steps; "
                             f"{lt_iter.loc[a, 'IQM']:.3f} at iteration {LBi}.")
        for a in [x for x in LLM_ARMS if x != 'GraphGPO']:
            for name, t in (('env steps', lt_steps), ('iterations', lt_iter)):
                v = verdict(t, a, 'GraphGPO')
                if v: lines.append(f"- `{a}` vs `GraphGPO` at equal {name}: {v[0]:+.3f} → **{v[2]}**")
        md_llm = '\n'.join(lines)
        display(Markdown(md_llm))
        with open('h2lab_v2_summary.md', 'a') as f:
            f.write('\n\n' + md_llm + '\n')
        print('Appended to h2lab_v2_summary.md')

## 8. The port: verl-agent on ALFWorld (separate notebook)
`ProgressTree.zip` is a recipe for [verl-agent](https://github.com/langfengQ/verl-agent) that sits next to its GraphGPO recipe without editing it; `ProgressTree_ALFWorld_Colab.ipynb` runs it on one Colab GPU (use a fresh runtime: it installs verl-agent's pinned stack).

| arm | rollouts | credit |
|---|---|---|
| `graphgpo` | G per task (upstream) | GraphGPO (upstream) |
| `progress` | G per task | progress credit, leave-one-out baselines |
| `progress_tree` | + branches from the most-visited states | progress credit (LOO) — the concept |
| `tree_grpo` | + branches from random positions | Tree-GRPO-style intra + inter-tree advantages (proxy) |
| `tree_graphgpo` | same branches as `progress_tree` | GraphGPO (attribution) |

- **Branching in a real environment:** a branch restores its parent's state by replaying the parent's first *t* environment actions on a private copy of the same ALFWorld game (deterministic), and the env manager rebuilds the prompt history from a snapshot. Every restore is checked against the logged state.
- **Verified on CPU** (no GPU where it was built): the port's estimator reproduces this notebook's `progress_loo_advantages` exactly on 24 toy batches; on real ALFWorld through verl-agent's own collector, env manager and Ray workers (with a mock LLM), every restore matched and every branch's first prompt was token-for-token its parent's prompt at the branch step; the invalid-action penalty enters exactly as in GraphGPO's episode term. Not verified: a GPU training run — the notebook starts with a GPU preflight and a 15-minute smoke run.
- **Protocol:** each arm trains for the same wall-clock hours (validation excluded) on the same game sequence per seed; minibatches are sized in samples (verl-agent's default), so branches add optimizer steps in proportion to the data. `compare_runs.py` reports success at the common budget, AUC, IQM over seeds and paired differences with bootstrap CIs. Cost: 6 h per arm and seed on one GPU; start with `graphgpo` vs `progress_tree` × 3 seeds.
- **WebShop** is not covered: branching needs a replayable environment and is implemented for ALFWorld; `progress` (no tree) works for any verl-agent environment.

## 9. Summary of findings
Generated from whatever ran in this session; sections that weren't run are skipped. Also saved to `h2lab_v2_summary.md`.

In [ ]:
# ===== Summary of findings (research context + this session's v2 results) =====
import numpy as np, pandas as pd
from IPython.display import Markdown, display

def _md_table(df, floatfmt=3):
    """Render a DataFrame as a Markdown table, formatting by column dtype."""
    def fmt(v, is_int):
        if isinstance(v, tuple):
            return '(' + ', '.join('—' if pd.isna(x) else f'{x:.3f}' for x in v) + ')'
        if v is None or (isinstance(v, (float, np.floating)) and np.isnan(v)):
            return '—'
        if isinstance(v, (bool, np.bool_)):
            return 'yes' if v else 'no'
        if is_int:
            return f'{int(v):,}'
        if isinstance(v, (float, np.floating)):
            return f'{v:.{floatfmt}f}'
        return str(v)
    cols = list(df.columns)
    is_int = {c: pd.api.types.is_integer_dtype(df[c]) for c in cols}
    lines = ['| ' + ' | '.join([df.index.name or ''] + [str(c) for c in cols]) + ' |',
             '|' + '---|' * (len(cols) + 1)]
    for idx in df.index:
        lines.append('| ' + ' | '.join([str(idx)] + [fmt(df.at[idx, c], is_int[c]) for c in cols]) + ' |')
    return '\n'.join(lines)

G = globals()
have = lambda *names: all(n in G for n in names)
S = []
fmt_v = lambda v: f"{v[0]:+.3f} → **{v[2]}**" if v else 'n/a'

# ---------------------------------------------------------------- 1. Context
S.append("""# Findings summary: v2.2 (progress-credited tree rollouts)

## 1. How we got here
- **Survey (mid-2025 to Nov 2026).** About 40 RL concepts reviewed; agentic step-level credit assignment was the clearest open gap.
- **Hybrid review.** Several cited foundations of the proposed hybrids were miscited or unverifiable (e.g. "Sony US 12,276,511 B2" is an Aptiv routing patent; SHARPO could not be found). Hybrid II (segment → state graph → Monte Carlo at the graph's frontier) was chosen and merged with H (exploration from restored states).
- **v1 result (full mode, 5 seeds).** Frontier MC improved credit on frontier steps (Spearman +0.17 vs GraphGPO) but not learning: at equal environment steps every arm that spent extra steps per iteration lost to plain GraphGPO (0.225), including frontier MC (0.168) and GraphGPO with 2× group size (0.178).
- **Diagnosis.** (1) On frontier steps GraphGPO's step advantage is zero by construction, leaving GRPO's whole-episode blame. (2) Re-linking frontier states through shortest paths discarded the success rates the probes measured: even MC at every state barely moved GraphGPO's gradient alignment (cosine 0.365 → 0.393), while the same probes through progress credit reached 0.547. (3) Probe steps were thrown away instead of trained on.
- **v2 design.** Branch the group into a tree from selected visited states, train on every branch, pool outcomes over merged states, and use progress credit V(s′) − V(s), which redistributes the GRPO advantage to the steps where success probability changed. Closest prior art: Tree-GRPO (random expansion, no cross-branch state merging), VinePPO (per-step MC values), PAV (progress as step reward), GraphGPO (merged state graph).
- **v2.1 result (full mode, 5 seeds, 2.4M env steps).** Progress+Tree(LOO) 0.51 and Progress+Tree 0.50 vs GraphGPO 0.39: both passed the pre-registered test, also with fixed updates per iteration. Behind GraphGPO when states merge less (merge noise 0.5 / 0.9).
- **v2.2 (this notebook).** A replication mode on new task pools; ALFWorld's merge rate measured under GraphGPO's state representation (0.44 → 0.91 of steps shared exactly, 0.68 → 0.93 with similarity merging, from a random to a 95%-expert policy); the crossover located (in pilot runs v2 led at noise 0 and trailed already at 0.15); sparse-graph remedies tested (none closed the gap); and a verl-agent port for ALFWorld, verified on CPU.
""")

# ---------------------------------------------------------------- 2. Kill test / branch diagnostics
sec = ["## 2. Kill test and branch diagnostics"]
if have('tr'):
    g = tr[tr.arm == 'GraphGPO']
    if len(g):
        fm = g.groupby('iter').frontier_moving_step_frac.mean()
        sec.append(f"- GraphGPO moving steps that start in a frontier state: **{fm.mean():.0%}** on average; above 5% in "
                   f"**{(fm > 0.05).mean():.0%}** of iterations → **{'PASS' if (fm > 0.05).mean() > 0.5 else 'KILL'}**.")
    for arm in [a for a in tr.arm.unique() if 'Tree' in a]:
        d = tr[tr.arm == arm]
        if 'branch_nodes' in d and d.branch_nodes.notna().any():
            tot = (d.branch_nodes + d.branch_frozen_skipped + d.branch_dead_skipped).mean()
            per_iter_env = d.env_steps.max() / max(d.iter.max(), 1)
            sec.append(f"- `{arm}`: {d.branch_nodes.mean():.0f} branches per iteration ({d.branch_steps.mean() / per_iter_env:.0%} of its env steps); "
                       f"{d.branch_frozen_skipped.mean() / tot:.0%} of candidate states skipped as absorbing, {d.branch_dead_skipped.mean() / tot:.0%} as cached dead ends; "
                       f"branch success rate {d.branch_success.mean():.0%}.")
else:
    sec.append("_Main experiment not run in this session._")
S.append('\n'.join(sec))

# ---------------------------------------------------------------- 3. Main result
sec = ["## 3. Main result (IQM eval success across seeds, 95% bootstrap CI)"]
main_v = {}
if have('t_steps', 'B_steps', 'verdict'):
    t = t_steps[['IQM', 'CI95', 'n']].copy(); t.index.name = 'arm'; t['IQM'] = t['IQM'].astype(float); t['n'] = t['n'].astype(int)
    sec.append(f"**At equal environment steps ({B_steps:,.0f}), same minibatch size for every arm — primary test:**\n\n" + _md_table(t))
    for f in [a for a in ['Progress+Tree', 'Progress+Tree(LOO)'] if a in t_steps.index]:
        for b in ['GraphGPO', 'GraphGPO(2xG)']:
            v = verdict(t_steps, f, b); main_v[(f, b)] = v
            if v: sec.append(f"- `{f}` vs `{b}`: {fmt_v(v)}")
    if have('t_iter', 'B_iter'):
        ti = t_iter[['IQM']].copy(); ti.index.name = 'arm'
        sec.append(f"\n**At equal iterations ({B_iter})** (arms with more samples per iteration also take more optimizer steps here):\n\n" + _md_table(ti))
    if have('sweep'):
        sw = pd.DataFrame(sweep); sw.index.name = 'arm'
        sec.append("\n**At equal total cost = env steps + per-iteration overhead:**\n\n" + _md_table(sw))
else:
    sec.append("_Main experiment not run in this session._")
S.append('\n\n'.join(sec))

# ---------------------------------------------------------------- 4. Attribution
if have('t_steps', 'verdict'):
    pairs = [('Data effect (graph credit): tree vs flat G', 'Tree+GraphGPO', 'GraphGPO'),
             ('Data effect (graph credit): tree vs flat 2G', 'Tree+GraphGPO', 'GraphGPO(2xG)'),
             ('Credit effect on the tree: progress vs graph', 'Progress+Tree', 'Tree+GraphGPO'),
             ('Credit effect on the tree: progress vs outcome-only', 'Progress+Tree', 'Tree+GRPO'),
             ('Branch-point effect: smart vs random', 'Progress+Tree', 'Progress+Tree(random)'),
             ('Baseline effect: leave-one-out vs telescoping', 'Progress+Tree(LOO)', 'Progress+Tree')]
    rows = {}
    for name, a, b in pairs:
        v = verdict(t_steps, a, b)
        if v:
            rows[name] = dict(comparison=f'{a} − {b}', delta=float(v[0]), CIs_separated=bool(v[1]))
    if rows:
        t = pd.DataFrame(rows).T; t.index.name = 'effect (equal env steps)'; t['delta'] = t['delta'].map(lambda v: f'{float(v):+.3f}')
        S.append("## 4. Attribution\n\n" + _md_table(t))

# ---------------------------------------------------------------- 5. Update-setting check
reg = {}
if have('rt_steps', 'rt_iter', 'rB_steps', 'rB_iter'):
    sec = ["## 5. Update-setting check (equal optimizer steps per iteration)"]
    for f in [a for a in ['Progress+Tree', 'Progress+Tree(LOO)'] if a in rt_steps.index]:
        for name, t, B in (('env steps', rt_steps, f'{rB_steps:,.0f}'), ('iterations', rt_iter, str(rB_iter))):
            v = verdict(t, f, 'GraphGPO'); reg[(f, name)] = v
            sec.append(f"- {f}, equal {name} ({B}): {t.loc[f, 'IQM']:.3f} vs GraphGPO {t.loc['GraphGPO', 'IQM']:.3f} → {fmt_v(v)}")
    S.append('\n'.join(sec))

# ---------------------------------------------------------------- 6. Credit study
cos_gain = None
if have('cs') and len(cs):
    late = cs[cs.snapshot > 0] if (cs.snapshot > 0).any() else cs
    tab = late.groupby(['estimator', 'subset']).cosine.mean().unstack()
    order = [e for e in ['GRPO', 'GraphGPO', 'GraphGPO+FrontierMC', 'GraphGPO+VineAll', 'Progress', 'Progress+VineAll',
                         'Tree+GRPO', 'Tree+GraphGPO', 'Progress+Tree', 'Progress+Tree(LOO)'] if e in tab.index]
    ext = late[late.subset == 'all'].groupby('estimator').extra_env_steps.mean()
    t = pd.DataFrame({'cosine all': tab.loc[order, 'all'], 'cosine frontier': tab.loc[order, 'frontier'],
                      'cosine linked': tab.loc[order, 'linked'], 'extra env steps / batch': ext.loc[order].round().astype(int)})
    t.index.name = 'estimator'
    sec = ["## 6. Credit accuracy (gradient alignment with ground truth)",
           f"Averaged over snapshots {sorted(late.snapshot.unique().tolist())}:\n\n" + _md_table(t)]
    for f in [a for a in ['Progress+Tree', 'Progress+Tree(LOO)'] if a in tab.index]:
        if 'GraphGPO' in tab.index:
            gain = tab.loc[f, 'all'] - tab.loc['GraphGPO', 'all']
            cos_gain = gain if cos_gain is None else max(cos_gain, gain)
            sec.append(f"- {f} vs GraphGPO, all steps: **{gain:+.3f}** cosine.")
    if {'GraphGPO+VineAll', 'Progress+VineAll'} <= set(tab.index):
        sec.append(f"- Same probes, different channel: Progress+VineAll {tab.loc['Progress+VineAll', 'all']:.3f} vs GraphGPO+VineAll "
                   f"{tab.loc['GraphGPO+VineAll', 'all']:.3f}.")
    S.append('\n'.join(sec))

# ---------------------------------------------------------------- 7. Stress test
if have('st') and len(st):
    pv = st.pivot(index='arm', columns='merge_noise', values='IQM'); pv.columns = [f'noise={c}' for c in pv.columns]; pv.index.name = 'arm'
    sec = [f"## 7. Sparse-graph stress test (IQM success at {P['stress_budget']:,} env steps)", _md_table(pv)]
    for f in [a for a in ['Progress+Tree', 'Progress+Tree(LOO)'] if a in pv.index]:
        if 'GraphGPO' in pv.index and pv.shape[1] > 1:
            gaps = ', '.join(f"{c}: {v:+.3f}" for c, v in (pv.loc[f] - pv.loc['GraphGPO']).items())
            sec.append(f"- {f} − GraphGPO: {gaps}")
    S.append('\n\n'.join(sec))

# ---------------------------------------------------------------- 7b. Merge-rate calibration (v2.2)
if have('ALFWORLD', 'RUN_CALIB') and RUN_CALIB and 'toy noise ≈ (exact)' in ALFWORLD:
    t = ALFWORLD.set_index('policy')[['success', 'exact', 'similarity', 'toy noise ≈ (exact)', 'toy noise ≈ (similarity)']].copy()
    t.index.name = 'ALFWorld policy'
    S.append("## 7b. Where ALFWorld sits on the toy's merge-noise scale\n"
             "Share of steps on a state another rollout of the group also visited (ALFWorld measured with GraphGPO's anchors; "
             "toy noise read off this session's pooled calibration curve; the toy's share barely depends on training success):\n\n" + _md_table(t, 2))

# ---------------------------------------------------------------- 7c. Sparse-graph remedies (v2.2)
if have('sw_tab', 'pv_sw') and len(sw_tab):
    pv = pv_sw.copy(); pv.columns = [f'noise={c}' for c in pv.columns]; pv.index.name = 'variant'
    order = [r for r in ['GraphGPO', 'Tree+GraphGPO', 'P+T(LOO) smart32', 'Progress(LOO)', 'P+T(LOO) smart8', 'P+T(LOO) smart8 K4',
                         'P+T(LOO) spread32', 'P+T(critic) smart32', 'P+T(critic) spread32'] if r in pv.index]
    sec = [f"## 7c. Sparse-graph remedies and branch budget (IQM success at {B_sw:,} env steps)", _md_table(pv.loc[order])]
    if have('sw_ver') and len(sw_ver):
        best = {}
        for q in sorted(sw_ver.noise.unique()):
            d = sw_ver[(sw_ver.noise == q) & (sw_ver.vs == 'GraphGPO')].sort_values('delta', ascending=False)
            if len(d):
                r = d.iloc[0]
                sec.append(f"- noise {q}: best remedy vs GraphGPO `{r.variant}` {r.delta:+.3f} → **{r.verdict}**")
            d2 = sw_ver[(sw_ver.noise == q) & (sw_ver.vs == 'P+T(LOO) smart32') & (sw_ver.verdict == 'PASS')]
            for _, r in d2.iterrows():
                sec.append(f"  - `{r.variant}` beats the v2.1 default at noise {q}: {r.delta:+.3f}")
    S.append('\n\n'.join(sec))

# ---------------------------------------------------------------- 8. Ablations / LLM
if have('ab') and len(ab) and have('at_budget', 'B_steps'):
    abe = ab[ab.eval_success.notna()]
    B_ab = min(B_steps, abe.groupby(['tag', 'seed']).env_steps.max().min())
    rows = {tag: dict(IQM=iqm(at_budget(d, 'env_steps', B_ab).success.values)) for tag, d in abe.groupby('tag')}
    if have('ev'):
        rows['(main) Progress+Tree'] = dict(IQM=iqm(at_budget(ev[ev.arm == 'Progress+Tree'], 'env_steps', B_ab).success.values))
    t = pd.DataFrame(rows).T; t.index.name = 'Progress+Tree variant'; t['IQM'] = t['IQM'].astype(float)
    S.append(f"## 8. Ablations (equal env steps, {B_ab:,.0f})\n\n" + _md_table(t.sort_values('IQM', ascending=False)))
if have('md_llm'):
    S.append(md_llm)
elif have('lev') and len(lev):
    rows = {}
    for arm, d in lev.groupby('arm'):
        d = d.sort_values('env_steps')
        rows[arm] = dict(final_success=float(d.eval_success.iloc[-1]), iterations=int(d.iter.iloc[-1]), env_steps=int(d.env_steps.iloc[-1]))
    t = pd.DataFrame(rows).T; t.index.name = 'arm'
    t['final_success'] = t['final_success'].astype(float); t['iterations'] = t['iterations'].astype(int); t['env_steps'] = t['env_steps'].astype(int)
    S.append("## 9. LLM track (Qwen2.5-0.5B-Instruct + LoRA, 1 seed, equal env-step budget)\n\n" + _md_table(t) +
             "\n\n- One seed: shows the pipeline works at LLM scale, not a statistically valid comparison.")

# ---------------------------------------------------------------- Bottom line
bl = ["## Bottom line"]
for f in [a for a in ['Progress+Tree', 'Progress+Tree(LOO)'] if (a, 'GraphGPO') in main_v]:
    v_g, v_2g = main_v.get((f, 'GraphGPO')), main_v.get((f, 'GraphGPO(2xG)'))
    if v_g is None:
        continue
    if v_g[2] == 'PASS' and (v_2g is None or v_2g[2] == 'PASS'):
        bl.append(f"- **`{f}` passes the pre-registered test at equal environment steps** (same minibatch size for all arms), against GraphGPO"
                  + (" and the budget-matched GraphGPO(2xG)." if v_2g else "."))
    elif v_g[0] > 0:
        bl.append(f"- **`{f}` is ahead of GraphGPO at equal environment steps ({v_g[0]:+.3f}) but the pre-registered test is not met** (needs ≥ +0.03 with separated CIs).")
    else:
        bl.append(f"- **`{f}` does not beat GraphGPO at equal environment steps here ({v_g[0]:+.3f}).**")
for f in ['Progress+Tree', 'Progress+Tree(LOO)']:
    ve, vi = reg.get((f, 'env steps')), reg.get((f, 'iterations'))
    if ve:
        head = 'Robust to minibatching' if ve[2] == 'PASS' else 'Minibatching matters'
        tail = '' if ve[2] == 'PASS' else ' In a real trainer, size minibatches in samples so branches add updates in proportion to the data.'
        bl.append(f"- **{head} ({f}).** With equal optimizer steps per iteration: {ve[0]:+.3f} vs GraphGPO at equal env steps"
                  + (f", {vi[0]:+.3f} at equal iterations" if vi else '') + '.' + tail)
if have('st') and len(st):
    pv2 = st.pivot(index='arm', columns='merge_noise', values='IQM')
    if 'GraphGPO' in pv2.index and pv2.shape[1] > 1:
        hi = pv2.columns.max()
        for f in [a for a in ['Progress+Tree', 'Progress+Tree(LOO)'] if a in pv2.index]:
            gap = pv2.loc[f, hi] - pv2.loc['GraphGPO', hi]
            bl.append(f"- **Sparse graphs ({f}):** {gap:+.3f} vs GraphGPO at merge noise {hi}"
                      + (" — the gains depend on states merging (where ALFWorld sits: section 7b)." if gap < 0 else "."))
if cos_gain is not None:
    bl.append(f"- **Credit quality:** Progress+Tree's advantages align better with ground truth than GraphGPO's ({cos_gain:+.3f} cosine)." if cos_gain > 0.02
              else f"- **Credit quality:** no clear gain in gradient alignment over GraphGPO ({cos_gain:+.3f} cosine).")
bl.append("""- **Next steps:**
  1. Run `MODE = 'replicate'` (new task pools, 10 seeds) to confirm the win where states merge.
  2. On ALFWorld, run the port's pilot first (GraphGPO vs Progress+Tree, 1 seed × 3 h each, similarity merging) and read the live merge rate (`main_shared_step_frac`). Run the multi-seed equal-GPU-hours comparison only if about 0.85 or more of steps are shared, or if Progress+Tree leads at 3 h.
  3. If ALFWorld confirms the toy, the next design problem is a merge-robust estimator: progress credit only between states that did merge and GraphGPO's credit elsewhere, or pooled values shrunk toward the last merged state's value when only one continuation passed through a state. Test it on the toy at noise 0.15–0.3 first.
- **Caveat:** a small grid-world with exact state restoration and cheap environment steps. Real agents merge fewer states, restore states by replaying action prefixes, and pay for every branch in LLM tokens.""")
S.append('\n'.join(bl))

report = '\n\n'.join(S)
display(Markdown(report))
with open('h2lab_v2_summary.md', 'w') as f:
    f.write(report)
print('Saved to h2lab_v2_summary.md')

## 10. Save results
CSVs and the summary are in the working directory. On Colab, this copies them to Google Drive.

In [ ]:
files = ['h2lab_v2_eval.csv', 'h2lab_v2_train.csv', 'h2lab_v2_credit.csv', 'h2lab_v2_stress.csv', 'h2lab_v2_calib.csv', 'h2lab_v2_sweep.csv',
         'h2lab_v2_llm.csv', 'h2lab_v2_summary.md']
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs('/content/drive/MyDrive/h2lab', exist_ok=True)
    for f in files:
        if os.path.exists(f): os.system(f'cp {f} /content/drive/MyDrive/h2lab/')
    print('Saved to MyDrive/h2lab')
else:
    print('Local run: files are in', os.getcwd())